In [ ]:
%%sql -r Uses
USE DATABASE STAR_DEV;
USE SCHEMA PROFILE;

# PATIENT MASTERING — CORE Methods

This notebook illustrates algorithmic logic that makes patient mastering work. Pipeline scaffolding, warehouses, RBAC, streams, tasks, audit plumbing, and DDL for infrastructure tables have not been considered.

What is here:

| Section Number | Title    | Description |
|--|--------------------------------|----------------------------------|  
|#1|  Standardization transforms    | improve raw values 'matchability'|
|#2|  Blocking key generation       | determine the candidate match space|
|#3|  Per-field scoring             | similarity computation|
|#4|  Composite scoring + decision  | scores become match/review/no-match|
|#5|  Survivorship                  | resolve conflicting records to one truth|
|#6|  Data quality rules            | detect and classify data issues| 
|#7|  Blocking quality metrics      | evaluate blocking key choices|
|#8|  Block skew diagnostics        | evaluate blocking key choice on compute|
|#9|  Singleton detection           | handle records not like th others|
|#10| Ground truth bootstrap        | find a set of highly confident 'true matches'|


# #1  STANDARDIZATION TRANSFORMS

**Input:**  Raw VARCHAR fields exactly as delivered by the source system

**Output:** Clean, comparable values + phonetic keys for blocking


In [ ]:
%%sql -r STANDARDIZE
CREATE OR REPLACE TEMPORARY TABLE STAN_NORM AS
WITH 
-- Identify Patients that don't appear in key fact tables
PATIENT_SRC AS ( 
    SELECT DISTINCT
        'APPOINTMENT' AS SRC,
        1 AS MARKER_FLAG,
        DIM_PATIENT_KEY,
    FROM
        STAR.DIM_MODEL.FACT_APPOINTMENT
    UNION
    SELECT
        'LEDGER',
        1,
        DIM_PATIENT_KEY
    FROM
        STAR.DIM_MODEL.FACT_BILLING_LEDGER
    UNION
    SELECT
        'ENCOUNTER',
        1,
        DIM_PATIENT_KEY
    FROM
        STAR.DIM_MODEL.FACT_ENCOUNTER_PROCEDURE
   UNION
    SELECT
        'RECALL',
        1,
        DIM_PATIENT_KEY
    FROM
        STAR.DIM_MODEL.FACT_RECALL 
   UNION
   SELECT
       'DIM_PATIENT',
       1,
       DIM_PATIENT_KEY
   FROM 
       STAR.DIM_MODEL.DIM_PATIENT 
),
TO_MATRIX AS(
    SELECT 
        *
    FROM 
        PATIENT_SRC
        PIVOT(COUNT(MARKER_FLAG) FOR SRC IN (ANY ORDER BY SRC))
    ORDER BY 
        DIM_PATIENT_KEY
),
-- Remove punctuation, control chars, zero-width spaces
CLEAN AS (
    SELECT
        DIM_PATIENT_KEY,
        FIRST_NAME,
        -- Identity: First Name
        --First Name Standardization
        --      Collapse multiple spaces
        --      Strip digits
        --      Strip non-sensical punctuation; keep hyphen, apostrophe, period
        --      Collapse repeated hyphens, apostrophes, periods
        --      Strip leading/trailing hyphens, apostrophes, periods
        --      Strip trailing period (initial like "J.")
        --      Proper-case each hyphen-separated segment
        normalize_first_name(FIRST_NAME) AS FIRST_NAME_cln,
        LAST_NAME,
        -- Identity: Last Name
        -- Last Name Standardization
        --      Collapse multiple spaces
        --      Strip digits
        --      Strip non-sensical punctuation; keep hyphen, apostrophe, period
        --      Collapse repeated hyphens, apostrophes, periods
        --      Strip leading/trailing hyphens, apostrophes, periods
        --      Split on hyphen, apply Mc / O' / standard proper-case per segment
        --      Normalize suffixes via CASE (avoids backreference regex)
        --      Proper-case each hyphen-separated segment
        normalize_last_name(LAST_NAME) AS LAST_NAME_cln,
        PATIENT_DOB,
        PATIENT_GENDER,
        PATIENT_ADDRESS1,
        -- Contact: Address1
        -- Uppercases 
        -- Trims
        -- Collapse whitespace
        -- Remove punctuation
        -- Exception: hyphens (for hyphenated house numbers like 12-14)
        cleanse_street_address(PATIENT_ADDRESS1) AS PATIENT_ADDRESS1_cln,
        PATIENT_CITY,
        INITCAP(TRIM(REGEXP_REPLACE(PATIENT_CITY,'[^a-zA-Z\\s\\-\\.\']',' '))) AS PATIENT_CITY_cln,
        PATIENT_STATE,
        TRIM(REGEXP_REPLACE(PATIENT_STATE,'[^A-Za-z\\s]',' ')) AS PATIENT_STATE_cln,
        SUBSTR(REGEXP_REPLACE(PATIENT_ZIPCODE, '[^0-9]', ''), 1, 5) AS PATIENT_ZIPCODE,
        MRN,
        CASE 
            WHEN MOBILE_NUMBER IS NOT NULL
            THEN TRIM(REGEXP_REPLACE(REPLACE(MOBILE_NUMBER,'-'),'^(1+|9+)$'))
            ELSE MOBILE_NUMBER
        END AS MOBILE_NUMBER,
        IDENTIFIER,
        ETL_CREATED_DATE
    FROM STAR.DIM_MODEL.DIM_PATIENT
),
STANDARDIZED AS (
    SELECT
        'UNMATCHED' AS MATCH_STATUS,
        DIM_PATIENT_KEY,
        FIRST_NAME,
        SUBSTR(FIRST_NAME,1,1) AS FN_INIT,
        LAST_NAME,
        SUBSTR(LAST_NAME,1,1) AS LN_INIT,
        SUBSTR(LAST_NAME,1,4) AS LN_SUB4,
        PATIENT_DOB,
        YEAR(PATIENT_DOB) AS YOB,
        PATIENT_GENDER,
        PATIENT_ADDRESS1_cln AS PATIENT_ADDRESS1,
        PATIENT_CITY_cln AS PATIENT_CITY,
        PATIENT_STATE,
        PATIENT_ZIPCODE,
        SUBSTR(PATIENT_ZIPCODE,1,3) AS ZIP3,
        MRN,
        IDENTIFIER,
        MOBILE_NUMBER,
        ETL_CREATED_DATE,
        FIRST_NAME_cln AS fn_std,
        LAST_NAME_cln AS ln_std,
        -- Phonetic blocking keys
        --  SOUNDEX maps names that sound alike to the same 4-character code.
        --  'SMITH', 'SMYTH', 'SMITHE' all produce S530.
        --  These columns are used in §2 to define blocking groups, not for
        --  scoring. They are intentionally lossy — that is the point.
        SOUNDEX(INITCAP(REGEXP_REPLACE(FIRST_NAME_cln, '\\s+', ' '))) AS FN_SDX,
        SOUNDEX(INITCAP(REGEXP_REPLACE(LAST_NAME_cln, '\\s+', ' '))) AS LN_SDX
        ,"'APPOINTMENT'","'LEDGER'","'ENCOUNTER'","'DIM_PATIENT'",
        
        -- Identity DQ  
        --  This is a primative proactive data-issue catcher - may go elsewhere.
        CASE 
          WHEN REGEXP_LIKE(FIRST_NAME, '(Blockittest|DUPLICATE|TEST|TESTING|DUMMY|SAMPLE|UNKNOWN|PATIENT|DEMO|ANONYMOUS|ANON|DO\\s?NOT\\s?USE|DELETE|FAKE|TEMP|PLACEHOLDER|NULL|N/A|NONE|XX+|ZZ+|QQ+|YY+)', 'i')
            OR REGEXP_LIKE(LAST_NAME, '(Blockittest|DUPLICATE|TEST|TESTING|DUMMY|SAMPLE|UNKNOWN|PATIENT|DEMO|ANONYMOUS|ANON|DO\\s?NOT\\s?USE|DELETE|FAKE|TEMP|PLACEHOLDER|NULL|N/A|NONE|XX+|ZZ+|QQ+|YY+)', 'i')
          THEN 1 
          ELSE 0 
          END AS FAUX_NAME,     
        CASE 
          WHEN REGEXP_LIKE(FIRST_NAME, '^\\s*$')
            OR REGEXP_LIKE(LAST_NAME, '^\\s*$')
          THEN 1 
          ELSE 0 
          END AS NAME_BLANK,
        CASE 
            WHEN PATIENT_DOB>=CURRENT_DATE()
            THEN 1
            ELSE 0
            END AS FUTURE_DOB,
        CASE
            WHEN PATIENT_DOB<'1901-01-31'::DATE
            THEN 1
            ELSE 0
            END AS FAUX_DATE,
        CASE 
          WHEN  REGEXP_LIKE(PATIENT_ZIPCODE,'\\d{5}')
          THEN 0
          ELSE 1
          END AS BAD_ZIP,
        CASE 
          WHEN PATIENT_GENDER IS NULL OR PATIENT_GENDER NOT IN ('Male','Female','Unknown','Other')
          THEN 1
          ELSE 0
          END AS BAD_GENDER,
        CASE 
          WHEN PATIENT_CITY IS NULL 
          THEN 1
          ELSE 0
          END AS BAD_CITY,
        CASE 
            WHEN PATIENT_STATE IS NULL
            THEN 1
            ELSE 0
            END AS BAD_STATE,
        CASE 
            WHEN PATIENT_ADDRESS1 IS NULL
            THEN 1
            ELSE 0
            END AS BAD_ADDRESS
    FROM CLEAN
    LEFT JOIN TO_MATRIX
    USING(DIM_PATIENT_KEY)
)
SELECT
    MATCH_STATUS,
    CURRENT_TIMESTAMP() AS STANDARDIZED_AT,
    DIM_PATIENT_KEY
    --Blocking Analysis Columns
    ,ETL_CREATED_DATE,LN_STD,LN_INIT,LN_SUB4,FN_STD,FN_INIT,FN_SDX,LN_SDX,PATIENT_DOB,YOB,PATIENT_GENDER,PATIENT_ADDRESS1,PATIENT_CITY,PATIENT_STATE,PATIENT_ZIPCODE,ZIP3,MOBILE_NUMBER,
    MRN,IDENTIFIER,
    --Ground Truth Pair Designation Columns
    FIRST_NAME,LAST_NAME,
    -- DQ Support
    FAUX_NAME,NAME_BLANK,FUTURE_DOB,FAUX_DATE,BAD_ZIP,BAD_GENDER,BAD_CITY,BAD_STATE,BAD_ADDRESS,
    ROUND(
          (CASE WHEN (FAUX_NAME = 1 OR NAME_BLANK = 1) THEN 50 ELSE 0 END
         + CASE WHEN (FUTURE_DOB = 1 OR FAUX_DATE = 1) THEN 30 ELSE 0 END
         + CASE WHEN BAD_GENDER = 1                    THEN 20 ELSE 0 END)
    , 2)
        AS DQ_SCORE_IDENTITY,

    ROUND(
          (CASE WHEN BAD_ADDRESS = 1   THEN 30 ELSE 0 END
         + CASE WHEN BAD_CITY = 1      THEN 25 ELSE 0 END
         + CASE WHEN BAD_STATE = 1     THEN 20 ELSE 0 END
         + CASE WHEN BAD_ZIP = 1       THEN 25 ELSE 0 END)
    , 2)   
        AS DQ_SCORE_CONTACT,
   ROUND( (DQ_SCORE_IDENTITY + DQ_SCORE_CONTACT) /2,2) AS DQ_SCORE_OVERALL
FROM 
    STANDARDIZED
--- Effectively Filter 'ghost patients'
WHERE "'APPOINTMENT'"=1 OR "'LEDGER'"=1 OR "'ENCOUNTER'"=1
;

In [ ]:
%%sql -r dataframe_42
SELECT TOP 5 * FROM STAN_NORM;

In [ ]:
%%sql -r dataframe_10
SELECT 'PAT KEY COUNT: ',COUNT(*) FROM STAN_NORM;

In [ ]:
%%sql -r dataframe_7
SELECT 
    FAUX_NAME,NAME_BLANK,FUTURE_DOB,FAUX_DATE,BAD_ZIP
    ,COUNT(*) AS FREQUENCY
FROM
    STAN_NORM
GROUP BY
   FAUX_NAME,NAME_BLANK,FUTURE_DOB,FAUX_DATE,BAD_ZIP
ORDER BY
    FREQUENCY DESC;

# #2  BLOCKING KEY GENERATION
***Goal:*** 

Generate only the candidate pairs worth scoring.
           Do NOT compare every record to every other record — that is O(n²) and will not complete on any real dataset.

***Strategy:*** 

Disjunctive blocking — a pair enters the candidate set if it matches on ANY of the blocking keys. More keys = higher recall
(Pairs Completeness) at a small efficiency cost (Reduction Ratio).

Passes are ordered: PRIMARY → FALLBACK → LAST_RESORT.
Records without any usable key end up in #9 (singletons).

Tuning these JOIN conditions is the highest-leverage decision in this entire system. See #7 for how to measure whether they are right.

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TEMPORARY TABLE BLOCKED AS
WITH
PRIMARY_BLOCK AS (
    SELECT
        A.DIM_PATIENT_KEY     AS REC_A,
        B.DIM_PATIENT_KEY     AS REC_B,
        'LNSDX_YOB'         AS BLOCK_PASS
    FROM STAN_NORM AS A
    JOIN STAN_NORM AS B
        ON  A.DIM_PATIENT_KEY      < B.DIM_PATIENT_KEY          -- No self join and ensures each pair appears once.
        AND A.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND B.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND A.LN_SDX  IS NOT NULL
        AND B.LN_SDX  IS NOT NULL
        AND A.YOB    IS NOT NULL
        AND B.YOB    IS NOT NULL
        AND A.LN_SDX = B.LN_SDX
        AND A.YOB = B.YOB 
),
FALLBACK_BLOCK AS (
    SELECT
        A.DIM_PATIENT_KEY     AS REC_A,
        B.DIM_PATIENT_KEY     AS REC_B,
        --   Last Name same SOUNDEX(last_name)
        --   --Catches: Smith / Smyth / Schmidt, transpositions, minor typos
        --   --DOB
        --   Initial Analysis shows 2,758,083 with 0.99 Reduction Rate and 1.0 Pairs Complete
        'LNSDX_ZIP'         AS BLOCK_PASS
    FROM STAN_NORM AS A
    JOIN STAN_NORM AS B
        ON  A.DIM_PATIENT_KEY      < B.DIM_PATIENT_KEY          
        AND A.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND B.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        -- at least one record has no usable last name key
        AND A.LN_SDX  IS NOT NULL
        AND B.LN_SDX  IS NOT NULL
        AND A.PATIENT_ZIPCODE      IS NOT NULL
        AND B.PATIENT_ZIPCODE      IS NOT NULL
        AND A.LN_SDX = B.LN_SDX
        AND A.PATIENT_ZIPCODE = B.PATIENT_ZIPCODE
),
LAST_RESORT_BLOCK AS (
    SELECT
        A.DIM_PATIENT_KEY     AS REC_A,
        B.DIM_PATIENT_KEY     AS REC_B,
        -- Initial Analysis shows 29,840,529 with 0.99 Reduction Rate and 1.0 Pairs Complete
        'ZIP_DOB'  AS BLOCK_PASS
    FROM STAN_NORM AS A
    JOIN STAN_NORM AS B
        ON  A.DIM_PATIENT_KEY      < B.DIM_PATIENT_KEY
        AND A.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND B.MATCH_STATUS       IN ('UNMATCHED', 'REVIEW')
        AND (A.LN_SDX  IS NULL OR B.LN_SDX  IS NULL)
        AND A.PATIENT_ZIPCODE    IS NOT NULL
        AND B.PATIENT_ZIPCODE    IS NOT NULL
        AND A.PATIENT_DOB        IS NOT NULL
        AND B.PATIENT_DOB        IS NOT NULL
        AND A.PATIENT_DOB          = B.PATIENT_DOB  
        AND A.PATIENT_ZIPCODE      = B.PATIENT_ZIPCODE
),
COMBINE_PASSES AS (
    SELECT * 
    FROM PRIMARY_BLOCK
    UNION ALL
    SELECT * 
    FROM FALLBACK_BLOCK
    UNION ALL
    SELECT * 
    FROM LAST_RESORT_BLOCK
)
--Prioritize Block Pairs
-- Keep Block Source for tracking purposes
SELECT 
    REC_A, 
    REC_B, 
    BLOCK_PASS
    FROM (
        SELECT *, 
           ROW_NUMBER() OVER (
               PARTITION BY REC_A, REC_B 
               ORDER BY 
                   CASE BLOCK_PASS 
                       WHEN 'LNSDX_ZIP' THEN 1
                       WHEN 'LNSDX_DOB' THEN 2
                       ELSE 3
                   END
           ) AS rn
    FROM COMBINE_PASSES
)
WHERE rn = 1;

In [ ]:
%%sql -r dataframe_9
SELECT COUNT(*) FROM
    (SELECT DISTINCT REC_A AS KEY FROM BLOCKED
     UNION
     SELECT DISTINCT REC_B FROM BLOCKED)
;

# #3  PER-FIELD SCORING

***Input:***  a candidate pair (REC_A, REC_B) from #2

***Output:*** a similarity score 0.0–1.0 per field

Each CASE expression is an independent scoring function.
They are combined in #4 using configurable weights.

     Notes on EDITDISTANCE():
       Snowflake's built-in function returns the minimum number of
       single-character edits (insertions, deletions, substitutions)
       needed to transform one string into another (Levenshtein distance).
       We normalize by the longer string's length to get a 0–1 similarity.
       This approximates Jaro-Winkler without a UDF:
         similarity = 1 - (edit_distance / max_length)
       For very short strings (length 1-2) this is unreliable; the
       NULLIF guard prevents division by zero.

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TEMPORARY TABLE FIELD_SCORES AS
WITH
SCORED AS (
    SELECT
        bp.REC_A,
        bp.REC_B,
        bp.BLOCK_PASS,
        
        -- SCORE_DOB 
        --  Exact match scores 1.0.
        CASE
            WHEN a.PATIENT_DOB IS NULL OR b.PATIENT_DOB IS NULL THEN 0.0
            WHEN a.PATIENT_DOB = b.PATIENT_DOB                  THEN 1.0
            ELSE 0.0
        END AS SCORE_DOB,

        /*
        -- SCORE_FNAME 
        --  Edit-distance similarity on the standardized first name.
        --  Handles: 'JOHN' vs 'JON', 'JOHNATHAN' vs 'JONATHAN'.
        --  Does NOT handle nickname → legal name ('BOB' vs 'ROBERT') —
        --  for that you need a nickname lookup table applied in #1.
        CASE
            WHEN a.fn_std IS NULL OR b.fn_std IS NULL THEN 0.0
            ELSE
                1.0 - (
                    EDITDISTANCE(a.fn_std, b.fn_std)::FLOAT
                    / NULLIF(GREATEST(LENGTH(a.fn_std),
                                      LENGTH(b.fn_std)), 0)
                )
        END AS SCORE_FNAME,
        */
        CASE
            WHEN a.fn_std IS NULL OR b.fn_std IS NULL THEN 0.0
            WHEN a.fn_std = b.fn_std THEN 1.0
            ELSE 0.0
        END AS SCORE_FNAME,
        
        -- SCORE_LNAME 
        --  Handles: 'SMITH' vs 'SMYTH', 'GONZALEZ' vs 'GONZALES'.
        --  SOUNDEX blocking in #2 already pre-filters for phonetic similarity,
        --  so most pairs reaching this score already share a Soundex code.
        --  This score refines within that phonetic bucket.
        CASE
            WHEN a.ln_std IS NULL OR b.ln_std IS NULL THEN 0.0
            ELSE
                1.0 - (
                    EDITDISTANCE(a.ln_std, b.ln_std)::FLOAT
                    / NULLIF(GREATEST(LENGTH(a.ln_std),
                                      LENGTH(b.ln_std)), 0)
                )
        END AS SCORE_lNAME,

        -- SCORE_GENDER 
        --  Binary. Low weight in #4 because gender can be recorded
        --  inconsistently across systems and updated over time.
        --  'U' (unknown) does not penalize — treated as missing, not mismatch.
        CASE
            WHEN a.PATIENT_GENDER IS NULL OR b.PATIENT_GENDER IS NULL              THEN 0.0
            WHEN a.PATIENT_GENDER = 'Unknown'    OR b.PATIENT_GENDER = 'Unknown'   THEN 0.0
            WHEN a.PATIENT_GENDER = b.PATIENT_GENDER                               THEN 1.0
            ELSE 0.0
        END AS SCORE_GENDER,

        -- SCORE_ZIP
        --  Exact 5-digit ZIP match only.
        --  Partial ZIP matching (first 3 digits) is intentionally not used:
        --  it would create massive false-positive rates in dense metro areas.
        --  ZIP is a corroborating signal, not a primary one.
        CASE
            WHEN a.PATIENT_ZIPCODE IS NULL OR b.PATIENT_ZIPCODE IS NULL THEN 0.0
            WHEN a.PATIENT_ZIPCODE = b.PATIENT_ZIPCODE                  THEN 1.0
            WHEN a.ZIP3 = b.ZIP3                                        THEN 0.8
            ELSE 0.0
        END AS SCORE_ZIP,

        -- SCORE_ZIP3
        --  Exact 3-digit ZIP match only.
        -- Here for informational purposes
        CASE
            WHEN a.ZIP3 IS NULL OR b.ZIP3 IS NULL THEN 0.0
            WHEN a.ZIP3 = b.ZIP3                  THEN 1.0
            ELSE 0.0
        END AS SCORE_ZIP3,

        -- SCORE ADDRESS1
        CASE
            WHEN a.PATIENT_ADDRESS1 IS NULL OR b.PATIENT_ADDRESS1 IS NULL THEN 0.0
            WHEN a.PATIENT_ADDRESS1 = b.PATIENT_ADDRESS1                  THEN 1.0
            ELSE 0.0
        END AS SCORE_ADDRESS,

        -- SCORE CITY
        CASE
            WHEN a.PATIENT_CITY IS NULL OR b.PATIENT_CITY IS NULL THEN 0.0
            WHEN a.PATIENT_CITY = b.PATIENT_CITY                  THEN 1.0
            ELSE 0.0
        END AS SCORE_CITY,
        
        -- SCORE STATE
        CASE
            WHEN a.PATIENT_STATE IS NULL OR b.PATIENT_STATE IS NULL THEN 0.0
            WHEN a.PATIENT_STATE = b.PATIENT_STATE                  THEN 1.0
            ELSE 0.0
        END AS SCORE_STATE,

        CASE 
            WHEN a.IDENTIFIER IS NULL OR b.IDENTIFIER IS NULL THEN 0.0
            WHEN a.IDENTIFIER = b.IDENTIFIER                 THEN 1.0
            ELSE 0.0
        END AS SCORE_IDEN,

    FROM BLOCKED AS bp
    JOIN STAN_NORM AS a ON a.DIM_PATIENT_KEY = bp.REC_A
    JOIN STAN_NORM AS b ON b.DIM_PATIENT_KEY = bp.REC_B 
)
SELECT * FROM SCORED;


In [ ]:
%%sql -r dataframe_6
/*
SELECT
    SCORE_DOB, SCORE_FNAME, SCORE_LNAME, SCORE_GENDER, SCORE_ZIP, SCORE_ADDRESS, SCORE_STATE, SCORE_CITY
    ,COUNT(*) AS FREQUENCY
FROM 
    FIELD_SCORES
GROUP BY
    SCORE_DOB, SCORE_FNAME, SCORE_LNAME, SCORE_GENDER, SCORE_ZIP, SCORE_ADDRESS, SCORE_STATE, SCORE_CITY
ORDER BY
    SCORE_DOB DESC, SCORE_FNAME DESC, SCORE_LNAME DESC, SCORE_GENDER DESC, SCORE_ADDRESS DESC, SCORE_ZIP DESC, SCORE_STATE DESC, SCORE_CITY DESC;
*/

## #4  COMPOSITE SCORING + THREE-TIER DECISION
***Input:***  Per-field scores from #3

***Output:*** Composite score 0.0–1.0 + decision label

Weights below reflect a standard healthcare MDM weighting scheme.

They MUST be externalized to a configuration table in production — do not hard-code them in application logic. Different source system combinations warrant different weight sets.

***Deterministic override:*** 
        If QQQQ OR PPP, the composite score is forced to 1.0 and the decision is AUTO_MATCH regardless of other field scores. Deterministic beats probabilistic. Always.

Three-tier decision:
       >= 0.85  → AUTO_MATCH     (no human needed)
       0.65–0.85 → REVIEW        (steward queue)
       < 0.65   → AUTO_NO_MATCH  (not the same patient)

These thresholds are calibrated against the ground truth set.
The values here are dynamic, not universal constants.

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TEMPORARY TABLE COMP_DECISION AS
SELECT
    REC_A,
    REC_B,
    BLOCK_PASS,
    -- Per-field score breakdown 
    --  Stored as a VARIANT so we can see why a pair scored as it did.
    OBJECT_CONSTRUCT(
        'dob',     SCORE_DOB,
        'fname',   SCORE_FNAME,
        'lname',   SCORE_LNAME,
        'gender',  SCORE_GENDER,
        'address', SCORE_ADDRESS,
        'zip',     SCORE_ZIP,
        'state',   SCORE_STATE,
        'city',    SCORE_CITY
    ) AS SCORE_BREAKDOWN,
    -- Weighted composite score 
    --  Weights sum to 1.0
    --
    --  Weight rationale:
    --    DOB     0.3  — high specificity, questionable corruption weight
    --    LNAME   0.30  — high specificity, moderate corruption (spelling, marriage)
    --    FNAME   0.20  — moderate specificity, higher corruption (nicknames)
    --    GENDER  0.05  — low discriminatory power, high recording inconsistency
    --    ZIP     0.05  — corroborating geographic signal, moderate staleness rate
    --    ADDRESS 0.00  — corroborating geographic signal, moderate staleness rate
    --    STATE   0.00  — corroborating geographic signal, moderate staleness rate
    --    CITY    0.00  — corroborating geographic signal, moderate staleness rate
    
    ROUND(
          SCORE_DOB     * 0.35
        + SCORE_LNAME   * 0.30
        + SCORE_FNAME   * 0.20
        + SCORE_GENDER  * 0.10
        + SCORE_ZIP     * 0.05
        + SCORE_ADDRESS * 0.00
        + SCORE_STATE   * 0.00
        + SCORE_CITY    * 0.00
    , 4) AS COMPOSITE_SCORE,
    -- Final score after consideration of deterministic override (intially this is redundant)
    CASE
        WHEN 
              SCORE_DOB     = 1
          AND SCORE_LNAME   = 1
          AND SCORE_FNAME   = 1
          AND SCORE_ZIP     = 1
          AND SCORE_ADDRESS = 1
          AND SCORE_STATE   = 1
          AND SCORE_CITY    = 1
          AND SCORE_GENDER  = 1
        THEN 1.0   -- deterministic win: ignore probabilistic score entirely
        ELSE COMPOSITE_SCORE
    END AS FINAL_SCORE,
    -- Three-tier match decision 
    CASE
        WHEN  
              SCORE_DOB     = 1
          AND SCORE_LNAME   = 1
          AND SCORE_FNAME   = 1
          AND SCORE_ZIP     = 1
          AND SCORE_ADDRESS = 1
          AND SCORE_STATE   = 1
          AND SCORE_CITY    = 1
          AND SCORE_GENDER  = 1               THEN 'TRUE_MATCH' -- deterministic win: ignore probabilistic score entirely
        WHEN
            FINAL_SCORE >= 0.85               THEN 'AUTO_MATCH'    -- high confidence
        WHEN 
            FINAL_SCORE >= 0.80               THEN 'REVIEW'        -- steward queue
        ELSE                                       'AUTO_NO_MATCH' -- different patients
    END AS MATCH_DECISION
FROM FIELD_SCORES
-- Only retain pairs at or above the review threshold, or deterministic matches.
-- Pairs below 0.75 with no deterministic signal are discarded here —
-- they will never appear in the review queue or contribute to a cluster.
WHERE 
    FINAL_SCORE>= 0.75
;

In [ ]:
%%sql -r dataframe_12
SELECT MATCH_DECISION,COUNT(*) AS FREQ FROM COMP_DECISION GROUP BY MATCH_DECISION;

In [ ]:
%%sql -r dataframe_18
WITH 
SAMPLE_ROWS AS (
    SELECT 
        REC_A,REC_B,FINAL_SCORE,
    FROM  
        COMP_DECISION
    WHERE 
          MATCH_DECISION IN ('REVIEW')
)
SELECT TOP 200
    REC_A,REC_B,FINAL_SCORE,
        a.LN_STD AS LN_STD_A,b.LN_STD AS LN_STD_B,
        a.FN_STD AS FN_STD_A,b.FN_STD AS FN_STD_B,
        a.PATIENT_DOB AS PATIENT_DOB_A,b.PATIENT_DOB AS PATIENT_DOB_B,
        a.PATIENT_GENDER AS PATIENT_GENDER_A,b.PATIENT_GENDER AS PATIENT_GENDER_b,
        a.PATIENT_CITY AS PATIENT_CITY_A, b.PATIENT_CITY AS PATIENT_CITY_B, 
        a.PATIENT_STATE AS PATIENT_STATE_A, b.PATIENT_STATE AS PATIENT_STATE_B, 
        a.PATIENT_ZIPCODE AS PATIENT_ZIPCODE_A, b.PATIENT_ZIPCODE PATIENT_ZIPCODE_B,
        a.MRN AS MRN_A, b.MRN AS MRN_b,
        a.IDENTIFIER AS IDENTIFIER_A, B.IDENTIFIER AS IDENTIFIER_B,
        a.PATIENT_ADDRESS1 AS PATIENT_ADDRESS1_A,b.PATIENT_ADDRESS1 AS PATIENT_ADDRESS1_B
FROM SAMPLE_ROWS AS chk
    JOIN STAN_NORM AS a ON a.DIM_PATIENT_KEY = chk.REC_A
    JOIN STAN_NORM AS b ON b.DIM_PATIENT_KEY = chk.REC_B 

# #5 Survivorship 
-- Resolve conflicting records to one truth


## PAIR INTAKE AND CLASSIFICATION
--     Take the #4 output and add the cluster context of each record.

--     Records may already belong to a cluster from a previous run.

--     Knowing this upfront drives every subsequent routing decision.

In [ ]:
%%sql -r dataframe_13
-- 'Historical' cluster membership — may be empty on first run
CREATE OR REPLACE TABLE MATCH_CLUSTERS (
    CLUSTER_ID              VARCHAR(40)     DEFAULT UUID_STRING() PRIMARY KEY,
    ENTERPRISE_PATIENT_ID   VARCHAR(40)     NOT NULL,
    DIM_PATIENT_KEY         VARCHAR(60)     NOT NULL,
    CLUSTER_CONFIDENCE      NUMBER(6,4),
    IS_ACTIVE               BOOLEAN         DEFAULT TRUE,
    LINKED_AT               TIMESTAMP_NTZ   DEFAULT CURRENT_TIMESTAMP(),
    LINKED_BY               VARCHAR(20)     DEFAULT 'AUTO',  -- AUTO / STEWARD / MERGE
    BATCH_ID                VARCHAR(40));

CREATE OR REPLACE TEMPORARY TABLE T_PAIR_INTAKE AS
WITH SCORED_PAIRS AS (
    -- Materialized output of #4 here.
    -- Required columns: REC_A, REC_B, FINAL_SCORE, MATCH_DECISION,
    --                   SCORE_BREAKDOWN, BLOCK_PASS
    SELECT * FROM COMP_DECISION
    WHERE MATCH_DECISION IN ('TRUE_MATCH','AUTO_MATCH')
),
EXISTING_CLUSTERS AS (
    -- Current cluster membership — may be empty on first run
    SELECT
        DIM_PATIENT_KEY,
        ENTERPRISE_PATIENT_ID,
        MAX(LINKED_AT) AS LINKED_AT
    FROM MATCH_CLUSTERS
    WHERE IS_ACTIVE = TRUE
    GROUP BY DIM_PATIENT_KEY, ENTERPRISE_PATIENT_ID
)
SELECT
    sp.REC_A,
    sp.REC_B,
    sp.FINAL_SCORE,
    sp.MATCH_DECISION,
    sp.SCORE_BREAKDOWN,
    sp.BLOCK_PASS,

    -- Existing EPI for each record — NULL means new to the system this run
    ea.ENTERPRISE_PATIENT_ID    AS EXISTING_EPI_A,
    eb.ENTERPRISE_PATIENT_ID    AS EXISTING_EPI_B,

    -- Classify the cluster situation this pair creates
    CASE
        -- Both records already in the same cluster → duplicate pair, skip
        WHEN ea.ENTERPRISE_PATIENT_ID IS NOT NULL
         AND ea.ENTERPRISE_PATIENT_ID = eb.ENTERPRISE_PATIENT_ID
                                                    THEN 'ALREADY_CLUSTERED'

        -- Both records new → form a new cluster
        WHEN ea.ENTERPRISE_PATIENT_ID IS NULL
         AND eb.ENTERPRISE_PATIENT_ID IS NULL        THEN 'NEW_CLUSTER'

        -- One record extends an existing cluster
        WHEN ea.ENTERPRISE_PATIENT_ID IS NOT NULL
         AND eb.ENTERPRISE_PATIENT_ID IS NULL        THEN 'EXTEND_CLUSTER_A'

        WHEN ea.ENTERPRISE_PATIENT_ID IS NULL
         AND eb.ENTERPRISE_PATIENT_ID IS NOT NULL    THEN 'EXTEND_CLUSTER_B'

        -- Both in different existing clusters → MERGE required
        WHEN ea.ENTERPRISE_PATIENT_ID IS NOT NULL
         AND eb.ENTERPRISE_PATIENT_ID IS NOT NULL
         AND ea.ENTERPRISE_PATIENT_ID
          != eb.ENTERPRISE_PATIENT_ID               THEN 'CROSS_CLUSTER_MERGE'

        ELSE 'UNKNOWN'
    END AS CLUSTER_SITUATION

FROM SCORED_PAIRS sp
LEFT JOIN EXISTING_CLUSTERS ea ON ea.DIM_PATIENT_KEY = sp.REC_A
LEFT JOIN EXISTING_CLUSTERS eb ON eb.DIM_PATIENT_KEY = sp.REC_B;


In [ ]:
%%sql -r dataframe_14
SELECT TOP 10 * FROM T_PAIR_INTAKE WHERE REC_A IN (4748790,7947117) OR REC_B IN (4748790,7947117);

##  CONNECTED-COMPONENT RESOLUTION

--     Problem: #4 gives pairs. Survivorship needs components.

--     If AUTO_MATCH pairs are:  A–B, B–C, C–D

--     Then the component is:   {A, B, C, D} → one ENTERPRISE_PATIENT_ID

--     SQL does not have a native graph traversal. The approach using

--     Snowflake is iterative label propagation:

--       - Assign each record its own label (its STD_RECORD_ID)

--       - Each iteration: a record adopts the smallest label of any neighbour it is directly connected to via an AUTO_MATCH pair

--       - Repeat until no labels change (convergence)

--       - Records sharing a label are in the same component


--     Convergence is guaranteed in at most k iterations where k is the

--     diameter of the largest component (typically 2–4 for patient data).

--     The WHILE loop below runs until stable.

--     This only runs on NEW_CLUSTER and EXTEND pairs from PAIR INTAKE AND CLASSIFICATION.

--     CROSS_CLUSTER_MERGE situations are handled separately in §D.

In [ ]:
%%sql -r dataframe_16
CREATE OR REPLACE TEMPORARY TABLE T_COMPONENT_EDGES AS
SELECT REC_A AS NODE_1, REC_B AS NODE_2, FINAL_SCORE
FROM T_PAIR_INTAKE
WHERE MATCH_DECISION IN ('AUTO_MATCH','TRUE_MATCH')
  AND CLUSTER_SITUATION IN ('NEW_CLUSTER', 'EXTEND_CLUSTER_A',
                             'EXTEND_CLUSTER_B');

-- Seed: each node is its own component
CREATE OR REPLACE TEMPORARY TABLE T_COMPONENT_LABELS AS
SELECT DISTINCT
    NODE                        AS DIM_PATIENT_KEY,
    NODE                        AS COMPONENT_LABEL,  -- smallest node in component
    0                           AS ITERATION
FROM (
    SELECT NODE_1 AS NODE FROM T_COMPONENT_EDGES
    UNION
    SELECT NODE_2 AS NODE FROM T_COMPONENT_EDGES
);


In [ ]:
%%sql -r dataframe_17
SELECT * FROM T_COMPONENT_LABELS WHERE DIM_PATIENT_KEY IN (4748790,7947117);

## Iterative label propagation

-- Each pass propagates the minimum label across all edges.

-- In Snowflake-SQL, pseudo-loop by repeats of a MERGE block a fixed number of times (safe for depth ≤ 6).

-- Four iterations handle components of depth up to 16 — sufficient for patient MDM where chain length is almost never > 4.

In [ ]:
%%sql -r dataframe_15
-- Iteration 1
MERGE INTO T_COMPONENT_LABELS tgt
USING (
    SELECT e.NODE_1 AS DIM_PATIENT_KEY,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL)) AS NEW_LABEL
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_1
    UNION ALL
    SELECT e.NODE_2,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL))
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_2
) src ON tgt.DIM_PATIENT_KEY = src.DIM_PATIENT_KEY
WHEN MATCHED AND src.NEW_LABEL < tgt.COMPONENT_LABEL
    THEN UPDATE SET COMPONENT_LABEL = src.NEW_LABEL, ITERATION = 1;
SELECT ITERATION, COUNT(*) FROM T_COMPONENT_LABELS GROUP BY 1;

-- Iteration 2 (same pattern — label propagates one more hop)
MERGE INTO T_COMPONENT_LABELS tgt
USING (
    SELECT e.NODE_1 AS  DIM_PATIENT_KEY,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL)) AS NEW_LABEL
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_1
    UNION ALL
    SELECT e.NODE_2,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL))
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_2
) src ON tgt.DIM_PATIENT_KEY = src.DIM_PATIENT_KEY
WHEN MATCHED AND src.NEW_LABEL < tgt.COMPONENT_LABEL
    THEN UPDATE SET COMPONENT_LABEL = src.NEW_LABEL, ITERATION = 2;
SELECT ITERATION, COUNT(*) FROM T_COMPONENT_LABELS GROUP BY 1;

-- Iteration 3
MERGE INTO T_COMPONENT_LABELS tgt
USING (
    SELECT e.NODE_1 AS  DIM_PATIENT_KEY,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL)) AS NEW_LABEL
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_1
    UNION ALL
    SELECT e.NODE_2,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL))
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_2
) src ON tgt.DIM_PATIENT_KEY = src.DIM_PATIENT_KEY
WHEN MATCHED AND src.NEW_LABEL < tgt.COMPONENT_LABEL
    THEN UPDATE SET COMPONENT_LABEL = src.NEW_LABEL, ITERATION = 3;
SELECT ITERATION, COUNT(*) FROM T_COMPONENT_LABELS GROUP BY 1;

-- Iteration 4
MERGE INTO T_COMPONENT_LABELS tgt
USING (
    SELECT e.NODE_1 AS  DIM_PATIENT_KEY,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL)) AS NEW_LABEL
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_1
    UNION ALL
    SELECT e.NODE_2,
           MIN(LEAST(la.COMPONENT_LABEL, lb.COMPONENT_LABEL))
    FROM T_COMPONENT_EDGES e
    JOIN T_COMPONENT_LABELS la ON la.DIM_PATIENT_KEY = e.NODE_1
    JOIN T_COMPONENT_LABELS lb ON lb.DIM_PATIENT_KEY = e.NODE_2
    GROUP BY e.NODE_2
) src ON tgt. DIM_PATIENT_KEY = src. DIM_PATIENT_KEY
WHEN MATCHED AND src.NEW_LABEL < tgt.COMPONENT_LABEL
    THEN UPDATE SET COMPONENT_LABEL = src.NEW_LABEL, ITERATION = 4;
SELECT ITERATION, COUNT(*) FROM T_COMPONENT_LABELS GROUP BY 1;

-- Convergence check: how many labels still changed on the last iteration?
-- If > 0, you have chains longer than 4 hops and need more iterations.
-- In practice this should always be 0 for patient MDM data.
SELECT
    COUNT_IF(ITERATION = 4)     AS LABELS_CHANGED_ON_FINAL_ITERATION,
    CASE
        WHEN COUNT_IF(ITERATION = 4) > 0
        THEN 'WARNING: chain depth > 4 — add more iterations'
        ELSE 'OK — label propagation converged'
    END                         AS CONVERGENCE_STATUS
FROM T_COMPONENT_LABELS;

In [ ]:
%%sql -r dataframe_24
SELECT TOP 10 * FROM T_COMPONENT_LABELS WHERE DIM_PATIENT_KEY IN (4748790,7947117) OR COMPONENT_LABEL IN (4748790,7947117);

In [ ]:
%%sql -r dataframe_19
SELECT ITERATION,COUNT(*) AS FREQ FROM T_COMPONENT_LABELS GROUP BY ITERATION

## EPI ASSIGNMENT

--     Each component from CONNECTED-COMPONENT RESOLUTION either:

--       (a) is entirely new → assign a fresh UUID as ENTERPRISE_PATIENT_ID

--       (b) contains at least one record already in an existing cluster → inherit the existing EPI (the one with the most records wins
--           if somehow two existing EPIs are in the same new component, which should only happen via the EXTEND paths from PAIR INTAKE AND CLASSIFICATION)

--     The result is a mapping: COMPONENT_LABEL → ENTERPRISE_PATIENT_ID

In [ ]:
%%sql -r dataframe_35
SELECT TOP 10 * FROM MATCH_CLUSTERS WHERE DIM_PATIENt_KEY IN (4748790,7947117)

In [ ]:
%%sql -r dataframe_23
CREATE OR REPLACE TEMPORARY TABLE T_EPI_ASSIGNMENT AS
WITH COMPONENT_EXISTING_EPI AS (
    -- For each component, find any pre-existing EPIs among its members
    SELECT
        cl.COMPONENT_LABEL,
        ex.ENTERPRISE_PATIENT_ID,
        COUNT(*) AS RECORDS_WITH_THIS_EPI
    FROM T_COMPONENT_LABELS cl
    JOIN MATCH_CLUSTERS ex ON ex.DIM_PATIENT_KEY = cl.DIM_PATIENT_KEY
                           AND ex.IS_ACTIVE = TRUE
    GROUP BY cl.COMPONENT_LABEL, ex.ENTERPRISE_PATIENT_ID
),
DOMINANT_EXISTING_EPI AS (
    -- If a component has multiple existing EPIs (shouldn't happen in
    -- clean data, but defend against it), pick the one with the most
    -- member records as the surviving EPI. The other becomes a merge source.
    SELECT
        COMPONENT_LABEL,
        FIRST_VALUE(ENTERPRISE_PATIENT_ID)
            OVER (PARTITION BY COMPONENT_LABEL
                  ORDER BY RECORDS_WITH_THIS_EPI DESC,
                           ENTERPRISE_PATIENT_ID ASC)   -- tie-break: lexicographic
            AS DOMINANT_EPI
    FROM COMPONENT_EXISTING_EPI
    QUALIFY ROW_NUMBER() OVER (PARTITION BY COMPONENT_LABEL
                               ORDER BY RECORDS_WITH_THIS_EPI DESC,
                                        ENTERPRISE_PATIENT_ID ASC) = 1
)
SELECT
    cl.COMPONENT_LABEL,
    COALESCE(
        de.DOMINANT_EPI,      -- existing EPI if any member was already clustered
        UUID_STRING()          -- fresh EPI for completely new components
    ) AS ENTERPRISE_PATIENT_ID,
    CASE
        WHEN de.DOMINANT_EPI IS NOT NULL THEN 'EXISTING'
        ELSE                                  'NEW'
    END AS EPI_TYPE
FROM (SELECT DISTINCT COMPONENT_LABEL FROM T_COMPONENT_LABELS) cl
LEFT JOIN DOMINANT_EXISTING_EPI de ON de.COMPONENT_LABEL = cl.COMPONENT_LABEL;

-- Diagnostic: how many new vs existing EPIs this run?
SELECT
    EPI_TYPE,
    COUNT(*) AS COMPONENT_COUNT
FROM T_EPI_ASSIGNMENT
GROUP BY EPI_TYPE;

In [ ]:
%%sql -r dataframe_11
SELECT * FROM T_EPI_ASSIGNMENT WHERE COMPONENT_LABEL IN (4748790,7947117)

##  CONFLICT DETECTION — CROSS-CLUSTER MERGES

--     CROSS_CLUSTER_MERGE pairs from PAIR INTAKE AND CLASSIFICATION mean an AUTO_MATCH pair whose members already belong to two different existing clusters.

--     This is the highest-risk operation in the pipeline:

--       - Merging two existing golden records into one destroys one of them

--       - If either golden record has been consumed downstream (shared with

--         a hospital system, billed against, used in a care plan), the merge

--         has external consequences

--       - This MUST be logged as a MERGE_EVENT before any data changes

--       - High-confidence merges (score >= 0.95 on a deterministic signal)

--         can be auto-approved; all others require steward sign-off

--     AUTO_MATCH pairs with CROSS_CLUSTER_MERGE situation are NOT added to

--     the cluster table in §E. They are logged here and routed to either:

--       (a) Auto-merge execution (§F) if score = 1.0 (deterministic)

--       (b) STEWARD_QUEUE (§G) if probabilistic, pending human approval

In [ ]:
%%sql -r dataframe_25
CREATE OR REPLACE TEMPORARY TABLE T_MERGE_CANDIDATES AS
SELECT
    pi.REC_A,
    pi.REC_B,
    pi.FINAL_SCORE,
    pi.MATCH_DECISION,
    pi.SCORE_BREAKDOWN,
    pi.EXISTING_EPI_A           AS SOURCE_EPI,  -- cluster to be absorbed
    pi.EXISTING_EPI_B           AS TARGET_EPI,  -- cluster to survive
    -- Snapshot the source cluster state before any changes
    OBJECT_CONSTRUCT(
        'epi',          pi.EXISTING_EPI_A,
        'record_count', src_size.MEMBER_COUNT,
        'members',      src_members.MEMBER_LIST,
        'captured_at',  CURRENT_TIMESTAMP()::VARCHAR
    )                           AS SOURCE_CLUSTER_SNAPSHOT,
    -- Auto-approve only deterministic matches (score = 1.0)
    -- Everything else goes to steward regardless of AUTO_MATCH decision
    CASE
        WHEN pi.FINAL_SCORE = 1.0 THEN 'AUTO_APPROVE'
        ELSE                           'STEWARD_REQUIRED'
    END                         AS MERGE_APPROVAL
FROM T_PAIR_INTAKE pi
JOIN (
    SELECT ENTERPRISE_PATIENT_ID, COUNT(*) AS MEMBER_COUNT
    FROM MATCH_CLUSTERS WHERE IS_ACTIVE = TRUE
    GROUP BY ENTERPRISE_PATIENT_ID
) src_size ON src_size.ENTERPRISE_PATIENT_ID = pi.EXISTING_EPI_A
JOIN (
    SELECT
        ENTERPRISE_PATIENT_ID,
        ARRAY_AGG(DIM_PATIENT_KEY) AS MEMBER_LIST
    FROM MATCH_CLUSTERS WHERE IS_ACTIVE = TRUE
    GROUP BY ENTERPRISE_PATIENT_ID
) src_members ON src_members.ENTERPRISE_PATIENT_ID = pi.EXISTING_EPI_A
WHERE pi.CLUSTER_SITUATION = 'CROSS_CLUSTER_MERGE'
  AND pi.MATCH_DECISION    = 'AUTO_MATCH';

-- Summary: how many merges, how many need steward sign-off?
-- Empty on initial run
SELECT
    MERGE_APPROVAL,
    COUNT(*)            AS MERGE_COUNT,
    MIN(FINAL_SCORE)    AS MIN_SCORE,
    MAX(FINAL_SCORE)    AS MAX_SCORE
FROM T_MERGE_CANDIDATES
GROUP BY MERGE_APPROVAL;

In [ ]:
%%sql -r dataframe_29
SELECT TOP 10 * FROM T_MERGE_CANDIDATES;

##  CLUSTER MEMBERSHIP POPULATION

Insert all resolved cluster memberships into MATCH_CLUSTERS.

     Categories of record:
       1. Records from new or extended components (CONNECTED-COMPONENT RESOLUTION & EPI ASSIGNMENT)
       2. Records absorbed into surviving clusters via auto-approved merges (CONFLICT DETECTION — CROSS-CLUSTER MERGES)
       

REVIEW pairs and STEWARD_REQUIRED merges are NOT inserted here — they wait in the steward queue until a human resolves them.

In [ ]:
%%sql -r dataframe_26
-- New and extended cluster members
MERGE INTO MATCH_CLUSTERS tgt
USING (
    SELECT
        cl.DIM_PATIENT_KEY,
        ea.ENTERPRISE_PATIENT_ID,
        -- Use the highest pair score involving this record as the confidence
        MAX(COALESCE(e1.FINAL_SCORE, e2.FINAL_SCORE)) AS CLUSTER_CONFIDENCE
    FROM T_COMPONENT_LABELS AS cl
    JOIN T_EPI_ASSIGNMENT   AS ea  
      ON ea.COMPONENT_LABEL = cl.COMPONENT_LABEL
    JOIN STAN_NORM AS s  
      ON s.DIM_PATIENT_KEY = cl.DIM_PATIENT_KEY
    LEFT JOIN T_COMPONENT_EDGES AS e1 
      ON e1.NODE_1 = cl.DIM_PATIENT_KEY
    LEFT JOIN T_COMPONENT_EDGES AS e2 
      ON e2.NODE_2 = cl.DIM_PATIENT_KEY
    GROUP BY 
        cl.DIM_PATIENT_KEY, 
        ea.ENTERPRISE_PATIENT_ID
) src ON tgt.DIM_PATIENT_KEY         = src.DIM_PATIENT_KEY
      AND tgt.ENTERPRISE_PATIENT_ID = src.ENTERPRISE_PATIENT_ID
      AND tgt.IS_ACTIVE              = TRUE

WHEN NOT MATCHED THEN INSERT (
    ENTERPRISE_PATIENT_ID,
    DIM_PATIENT_KEY,
    CLUSTER_CONFIDENCE,
    IS_ACTIVE,
    LINKED_BY
) VALUES (
    src.ENTERPRISE_PATIENT_ID,
    src.DIM_PATIENT_KEY,
    src.CLUSTER_CONFIDENCE,
    TRUE,
    'AUTO'
);

-- Auto-approved cross-cluster merges 
-- For each auto-approved merge: move all records from SOURCE_EPI to TARGET_EPI.
-- Deactivate old memberships first, then re-insert under the surviving EPI.

-- Step 1: deactivate source cluster members
UPDATE MATCH_CLUSTERS
SET    IS_ACTIVE  = FALSE,
       LINKED_BY  = 'MERGE'
WHERE  ENTERPRISE_PATIENT_ID IN (
    SELECT DISTINCT SOURCE_EPI
    FROM T_MERGE_CANDIDATES
    WHERE MERGE_APPROVAL = 'AUTO_APPROVE'
)
AND IS_ACTIVE = TRUE;

-- Step 2: re-insert those records under the surviving (target) EPI
INSERT INTO MATCH_CLUSTERS (
    ENTERPRISE_PATIENT_ID,
    DIM_PATIENT_KEY,
    CLUSTER_CONFIDENCE,
    IS_ACTIVE,
    LINKED_BY
)
SELECT
    mc.TARGET_EPI,
    old.DIM_PATIENT_KEY,
    mc.FINAL_SCORE,
    TRUE,
    'MERGE'
FROM T_MERGE_CANDIDATES mc
JOIN MATCH_CLUSTERS AS old
    ON  old.ENTERPRISE_PATIENT_ID = mc.SOURCE_EPI
    AND old.IS_ACTIVE              = FALSE    -- just deactivated above
    AND old.LINKED_BY              = 'MERGE'
WHERE mc.MERGE_APPROVAL = 'AUTO_APPROVE';



In [ ]:
%%sql -r dataframe_31
SELECT TOP 10 * FROM MATCH_CLUSTERS WHERE DIM_PATIENT_KEY IN (4748790,7947117);

## MERGE EVENT LOGGING

--     Every cluster merge — auto or steward-approved — must be logged

--     before any golden record is written or updated.

--     This is the non-negotiable audit trail for HIPAA and for the

--     ability to UNDO a bad merge later.


In [ ]:
%%sql -r dataframe_27
CREATE OR REPLACE TABLE MERGE_EVENTS (
    EVENT_ID                VARCHAR(40)     DEFAULT UUID_STRING() PRIMARY KEY,
    EVENT_TYPE              VARCHAR(20)     NOT NULL,   -- MERGE / SPLIT / RELINK
    SOURCE_EPI              VARCHAR(40),    -- cluster being absorbed
    TARGET_EPI              VARCHAR(40),    -- surviving cluster
    AFFECTED_STD_RECORD_IDS ARRAY,
    PERFORMED_BY            VARCHAR(40)     DEFAULT 'AUTO',
    PERFORMED_AT            TIMESTAMP_NTZ   DEFAULT CURRENT_TIMESTAMP(),
    BATCH_ID                VARCHAR(40),
    REASON                  VARCHAR(500),
    REQUIRES_REVIEW         BOOLEAN         DEFAULT FALSE,
    PREVIOUS_CLUSTER_STATE  VARIANT         -- snapshot of SOURCE_EPI cluster before merge
);

INSERT INTO MERGE_EVENTS (
    EVENT_TYPE,
    SOURCE_EPI,
    TARGET_EPI,
    AFFECTED_STD_RECORD_IDS,
    PERFORMED_BY,
    BATCH_ID,
    REASON,
    REQUIRES_REVIEW,
    PREVIOUS_CLUSTER_STATE
)
SELECT
    'MERGE'                         AS EVENT_TYPE,
    mc.SOURCE_EPI,
    mc.TARGET_EPI,
    -- Collect all STD_RECORD_IDs from the source cluster
    (SELECT ARRAY_AGG(DIM_PATIENT_KEY)
     FROM MATCH_CLUSTERS
     WHERE ENTERPRISE_PATIENT_ID = mc.SOURCE_EPI)
                                    AS AFFECTED_STD_RECORD_IDS,
    CASE mc.MERGE_APPROVAL
        WHEN 'AUTO_APPROVE' THEN 'AUTO'
        ELSE                     'PENDING_STEWARD'
    END                             AS PERFORMED_BY,
    1                            AS BATCH_ID,   -- Start with 1
    'AUTO_MATCH pair score=' || mc.FINAL_SCORE::VARCHAR
        || ' between ' || mc.REC_A || ' and ' || mc.REC_B
                                    AS REASON,
    CASE mc.MERGE_APPROVAL
        WHEN 'AUTO_APPROVE' THEN FALSE
        ELSE                     TRUE
    END                             AS REQUIRES_REVIEW,
    mc.SOURCE_CLUSTER_SNAPSHOT      AS PREVIOUS_CLUSTER_STATE
FROM T_MERGE_CANDIDATES mc;

In [ ]:
%%sql -r dataframe_28
SELECT TOP 10 * FROM MERGE_EVENTS;

## PRE-SURVIVORSHIP READINESS CHECKS

Before survivorship runs, validate that each cluster is in a state where a meaningful golden record can be produced.

     A cluster is NOT ready if:
       - It has fewer records than expected (a pairing went wrong)
       - All records in the cluster have a CRITICAL DQ issue on DOB
         or name (the survivorship winner will have an unusable field)
       - It has an unresolved open MERGE_EVENT (pending steward review)
       - Every record in the cluster is missing the same critical field
         (no amount of survivorship can fill a gap that exists everywhere)

     Clusters that fail readiness are flagged in GOLDEN_READINESS and excluded from the survivorship run. They surface in the steward
     dashboard for manual intervention.

In [ ]:
/*
 HYBRID FIELD-LEVEL STRATEGY
     Input:  match_clusters (active memberships) + std_patient_records
     Output: one golden record per ENTERPRISE_PATIENT_ID

     No source trust hierarchy is assumed. Each field is resolved
     independently using the strategy best suited to how that field
     behaves in the real world:

       MOST_FREQUENT  — field should be stable; disagreement = one source
                        has a data entry error; consensus is the truth.
                        Applied to: DOB, names, gender, race, ethnicity.

       MOST_RECENT    — field changes legitimately over time; the newest
                        value is more likely to be correct than the most
                        common one. Applied to: address, phone, email,
                        insurance.

       ANY_TRUE       — flag is set if any source asserts it. Applied to
                        DECEASED_FLAG — one source knowing the patient is
                        deceased is sufficient regardless of what others say.

     TIE-BREAKING
     When two values tie on frequency (common with two sources), the
     tie-break order is:
       1. Longest non-null string  — more complete is preferred
       2. Most recent              — newer is preferred over older
       3. Lexicographic ascending  — deterministic fallback

     CONFLICT FLAGGING
     A field is marked CONFLICTED when:
       - Exactly two distinct non-null values exist in the cluster AND
       - Both appear with equal frequency (true 1-1 tie, no majority)
     Conflicted fields are set to NULL in the golden record and logged
     in a companion GOLDEN_CONFLICTS table for steward resolution.
     A golden record with an honest NULL is safer than one that silently
     chose wrong.

     PATTERN
     Each field is a separate CTE using the same structure:
       1. GROUP BY (ENTERPRISE_PATIENT_ID, field_value) to get vote counts
       2. ROW_NUMBER() with strategy-appropriate ORDER BY to rank values
       3. Detect tie condition for conflict flagging
       4. Final SELECT picks RN = 1 from each CTE

     Adding a new field means copying one CTE block and changing three
     things: the column name, the ORDER BY strategy, and the JOIN below.
*/
 
WITH
CLUSTER_MEMBERS AS (
-- Base: all active cluster members 
    SELECT
        cl.ENTERPRISE_PATIENT_ID,
        s.DIM_PATIENT_KEY,
        --s.SOURCE_SYSTEM_ID, -- Useful in another version
        s.STANDARDIZED_AT,
        s.FN_STD,
        --s.MIDDLE_NAME_STD, -- Tentative
        s.LN_STD,
        --s.NAME_SUFFIX_STD, -- Tentative
        s.PATIENT_DOB,
        s.PATIENT_GENDER,
        --s.RACE_STD, -- Tentative
        --s.ETHNICITY_STD, -- Tentative
        s.PATIENT_ADDRESS1,
        --s.ADDRESS_LINE2_STD, -- Tentative
        s.PATIENT_CITY,
        s.PATIENT_STATE,
        s.PATIENT_ZIPCODE,
        --s.COUNTRY_STD, -- Tentative
        --s.PHONE_STD, -- Tentative
        --s.EMAIL_STD,  -- Tentative
        --s.INSURANCE_ID, -- Tentative
        --s.DECEASED_FLAG, -- Tentative
        --s.DECEASED_DATE, -- Tentative
        s.DQ_SCORE_OVERALL
    FROM match_clusters cl
    JOIN STAN_NORM AS s 
      ON s.DIM_PATIENT_KEY = cl.DIM_PATIENT_KEY
    WHERE cl.IS_ACTIVE = TRUE
),
CLUSTER_STATS AS (
-- Cluster-level stats
    SELECT
        ENTERPRISE_PATIENT_ID,
        COUNT(*)                            AS RECORD_COUNT,
        -- COUNT(DISTINCT SOURCE_SYSTEM_ID)    AS SOURCE_COUNT, -- May become relevant
        AVG(cl.CLUSTER_CONFIDENCE)          AS AVG_CONFIDENCE
    FROM match_clusters AS cl
    WHERE cl.IS_ACTIVE = TRUE
    GROUP BY ENTERPRISE_PATIENT_ID
), 
SOURCE_ID_BAG AS (
-- Source identifier bag 
    SELECT
        cl.ENTERPRISE_PATIENT_ID,
        OBJECT_AGG('KEYS', s.DIM_PATIENT_KEY) AS SOURCE_IDENTIFIERS
    FROM match_clusters AS cl
    JOIN STAN_NORM AS s 
    ON s.DIM_PATIENT_KEY = cl.DIM_PATIENT_KEY
    WHERE cl.IS_ACTIVE = TRUE
    GROUP BY cl.ENTERPRISE_PATIENT_ID
), 
/*
 MOST_FREQUENT fields — identity fields that should be stable
 Pattern for each:
   a) count occurrences of each distinct value per cluster
   b) detect a true tie (two values, equal counts) → CONFLICTED
   c) rank by frequency DESC, length DESC, recency DESC, alpha ASC
   d) pick RN = 1
*/
FNAME_VOTES AS (
-- FIRST_NAME 
    SELECT
        ENTERPRISE_PATIENT_ID,
        FN_STD,
        COUNT(*)                                AS FREQ,
        MAX(STANDARDIZED_AT)                    AS LATEST,
        -- Detect tie: exactly 2 distinct values with equal counts
        COUNT(*) = MIN(COUNT(*)) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        )
        AND COUNT(DISTINCT FN_STD) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        ) > 1                                   AS IS_TIED,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY
                COUNT(*)                DESC,   -- most frequent first
                LENGTH(FN_STD)  DESC,   -- longer = more complete
                MAX(STANDARDIZED_AT)    DESC,   -- more recent
                FN_STD          ASC     -- deterministic fallback
        )                                       AS RN
    FROM CLUSTER_MEMBERS
    WHERE FN_STD IS NOT NULL
    GROUP BY ENTERPRISE_PATIENT_ID, FN_STD
),
 
FNAME_SURVIVED AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        CASE WHEN IS_TIED THEN NULL ELSE FN_STD END AS FIRST_NAME,
        IS_TIED                                             AS FNAME_CONFLICTED
    FROM FNAME_VOTES WHERE RN = 1
),
/*
MIDDLE_NAME 
 Middle name is frequently absent in one source and present in another.
 Prefer non-null; among non-nulls use frequency then length.
*/
/*
MNAME_VOTES AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        MIDDLE_NAME_STD,
        COUNT(*)                                AS FREQ,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY
                COUNT(*)                DESC,
                LENGTH(MIDDLE_NAME_STD) DESC,
                MAX(STANDARDIZED_AT)    DESC,
                MIDDLE_NAME_STD         ASC
        )                                       AS RN
    FROM CLUSTER_MEMBERS
    WHERE MIDDLE_NAME_STD IS NOT NULL
    GROUP BY ENTERPRISE_PATIENT_ID, MIDDLE_NAME_STD
),
MNAME_SURVIVED AS (
    SELECT ENTERPRISE_PATIENT_ID, MIDDLE_NAME_STD AS MIDDLE_NAME
    FROM MNAME_VOTES WHERE RN = 1
),
*/
LNAME_VOTES AS (
-- LAST_NAME 
    SELECT
        ENTERPRISE_PATIENT_ID,
        LN_STD,
        COUNT(*)                                AS FREQ,
        COUNT(*) = MIN(COUNT(*)) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        )
        AND COUNT(DISTINCT LN_STD) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        ) > 1                                   AS IS_TIED,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY
                COUNT(*)                DESC,
                LENGTH(LN_STD)   DESC,
                MAX(STANDARDIZED_AT)    DESC,
                LN_STD           ASC
        )                                       AS RN
    FROM CLUSTER_MEMBERS
    WHERE LN_STD IS NOT NULL
    GROUP BY ENTERPRISE_PATIENT_ID, LN_STD
),
LNAME_SURVIVED AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        CASE WHEN IS_TIED THEN NULL ELSE LN_STD END AS LAST_NAME,
        IS_TIED                                            AS LNAME_CONFLICTED
    FROM LNAME_VOTES WHERE RN = 1
),
/*
SUFFIX_VOTES AS (
--  NAME_SUFFIX 
    SELECT
        ENTERPRISE_PATIENT_ID,
        NAME_SUFFIX_STD,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY COUNT(*) DESC, NAME_SUFFIX_STD ASC
        ) AS RN
    FROM CLUSTER_MEMBERS
    WHERE NAME_SUFFIX_STD IS NOT NULL
    GROUP BY ENTERPRISE_PATIENT_ID, NAME_SUFFIX_STD
),
SUFFIX_SURVIVED AS (
    SELECT ENTERPRISE_PATIENT_ID, NAME_SUFFIX_STD AS NAME_SUFFIX
    FROM SUFFIX_VOTES WHERE RN = 1
),
*/
/*
  DATE_OF_BIRTH 
 DOB should never legitimately change. Disagreement always means one
 source has an error. Most frequent value wins. Tie = conflict.
 Additional guard: exclude implausible values (year < 1900 or future)
 before voting so placeholder dates don't win by frequency.
*/
DOB_VOTES AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        PATIENT_DOB,
        COUNT(*)                                AS FREQ,
        COUNT(*) = MIN(COUNT(*)) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        )
        AND COUNT(DISTINCT PATIENT_DOB) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        ) > 1                                   AS IS_TIED,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY COUNT(*) DESC, PATIENT_DOB ASC
        )                                       AS RN
    FROM CLUSTER_MEMBERS
    WHERE PATIENT_DOB IS NOT NULL
      AND PATIENT_DOB > '1900-01-01'::DATE    -- exclude placeholder dates
      AND PATIENT_DOB < CURRENT_DATE()        -- exclude future dates
    GROUP BY ENTERPRISE_PATIENT_ID, PATIENT_DOB
), 
DOB_SURVIVED AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        CASE WHEN IS_TIED THEN NULL ELSE PATIENT_DOB END AS DATE_OF_BIRTH,
        IS_TIED                                            AS DOB_CONFLICTED
    FROM DOB_VOTES WHERE RN = 1
),
/* 
 GENDER 
 Exclude 'U' (unknown) from voting — it is absence of information,
 not a value to propagate. A real value beats U every time.
*/
GENDER_VOTES AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        PATIENT_GENDER,
        COUNT(*)                                AS FREQ,
        COUNT(*) = MIN(COUNT(*)) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        )
        AND COUNT(DISTINCT PATIENT_GENDER) OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
        ) > 1                                   AS IS_TIED,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY COUNT(*) DESC, PATIENT_GENDER ASC
        )                                       AS RN
    FROM CLUSTER_MEMBERS
    WHERE PATIENT_GENDER IS NOT NULL
      AND PATIENT_GENDER != 'U'
    GROUP BY ENTERPRISE_PATIENT_ID, PATIENT_GENDER
), 
GENDER_SURVIVED AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        CASE WHEN IS_TIED THEN NULL ELSE PATIENT_GENDER END AS GENDER,
        IS_TIED                                         AS GENDER_CONFLICTED
    FROM GENDER_VOTES WHERE RN = 1
),
/*
 RACE / ETHNICITY 
 Most frequent; no conflict flagging — race/ethnicity can legitimately
 differ across sources due to self-report vs. administrative coding.
 Prefer the most complete (longest) value on ties.
*/
/*
RACE_VOTES AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        RACE_STD,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY COUNT(*) DESC, LENGTH(RACE_STD) DESC, RACE_STD ASC
        ) AS RN
    FROM CLUSTER_MEMBERS
    WHERE RACE_STD IS NOT NULL
    GROUP BY ENTERPRISE_PATIENT_ID, RACE_STD
), 
RACE_SURVIVED AS (
    SELECT ENTERPRISE_PATIENT_ID, RACE_STD AS RACE
    FROM RACE_VOTES WHERE RN = 1
),
/*
/*
ETHNICITY_VOTES AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        ETHNICITY_STD,
        ROW_NUMBER() OVER (
            PARTITION BY ENTERPRISE_PATIENT_ID
            ORDER BY COUNT(*) DESC, LENGTH(ETHNICITY_STD) DESC, ETHNICITY_STD ASC
        ) AS RN
    FROM CLUSTER_MEMBERS
    WHERE ETHNICITY_STD IS NOT NULL
    GROUP BY ENTERPRISE_PATIENT_ID, ETHNICITY_STD
),
ETHNICITY_SURVIVED AS (
    SELECT ENTERPRISE_PATIENT_ID, ETHNICITY_STD AS ETHNICITY
    FROM ETHNICITY_VOTES WHERE RN = 1
),
*/
/*
 MOST_RECENT fields — contact fields that change legitimately over time
 Pattern: rank by STANDARDIZED_AT DESC within each cluster.
 No frequency voting — the most recent value wins outright.
 Tie-break: longest non-null string, then deterministic sort.
 No conflict flagging for contact fields — stale vs. current is not
 a conflict, it is just older data losing to newer data.

 
 ADDRESS (treat as a unit — street + city + state + ZIP together) 
 Splitting address across separate CTEs risks recombining fields from
 different records (street from one, ZIP from another). Keep them together
 by picking the most recent record that has a non-null street address,
 then taking all address subfields from that same record.
*/ 
ADDRESS_SURVIVED AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        PATIENT_ADDRESS1,
        --ADDRESS_LINE2_STD,
        PATIENT_CITY,
        PATIENT_STATE,
        PATIENT_ZIPCODE,
        --COUNTRY_STD
    FROM (
        SELECT
            ENTERPRISE_PATIENT_ID,
            PATIENT_ADDRESS1,
            --ADDRESS_LINE2_STD,
            PATIENT_CITY,
            PATIENT_STATE,
            PATIENT_ZIPCODE,
            --COUNTRY_STD,
            ROW_NUMBER() OVER (
                PARTITION BY ENTERPRISE_PATIENT_ID
                ORDER BY
                    STANDARDIZED_AT             DESC,   -- most recent record
                    LENGTH(PATIENT_ADDRESS1)    DESC,   -- more complete
                    PATIENT_ADDRESS1            ASC
            ) AS RN
        FROM CLUSTER_MEMBERS
        WHERE PATIENT_ADDRESS1 IS NOT NULL
    )
    WHERE RN = 1
),
/*
-- PHONE 
PHONE_SURVIVED AS (
    SELECT ENTERPRISE_PATIENT_ID, PHONE_STD
    FROM (
        SELECT
            ENTERPRISE_PATIENT_ID,
            PHONE_STD,
            ROW_NUMBER() OVER (
                PARTITION BY ENTERPRISE_PATIENT_ID
                ORDER BY
                    STANDARDIZED_AT         DESC,
                    LENGTH(PHONE_STD)       DESC,
                    PHONE_STD               ASC
            ) AS RN
        FROM CLUSTER_MEMBERS
        WHERE PHONE_STD IS NOT NULL
    )
    WHERE RN = 1
),
*/
/*
-- EMAIL 
EMAIL_SURVIVED AS (
    SELECT ENTERPRISE_PATIENT_ID, EMAIL_STD
    FROM (
        SELECT
            ENTERPRISE_PATIENT_ID,
            EMAIL_STD,
            ROW_NUMBER() OVER (
                PARTITION BY ENTERPRISE_PATIENT_ID
                ORDER BY
                    STANDARDIZED_AT         DESC,
                    LENGTH(EMAIL_STD)       DESC,
                    EMAIL_STD               ASC
            ) AS RN
        FROM CLUSTER_MEMBERS
        WHERE EMAIL_STD IS NOT NULL
    )
    WHERE RN = 1
),
*/
/*
-- INSURANCE_ID 
INSURANCE_SURVIVED AS (
    SELECT ENTERPRISE_PATIENT_ID, INSURANCE_ID
    FROM (
        SELECT
            ENTERPRISE_PATIENT_ID,
            INSURANCE_ID,
            ROW_NUMBER() OVER (
                PARTITION BY ENTERPRISE_PATIENT_ID
                ORDER BY STANDARDIZED_AT DESC, INSURANCE_ID ASC
            ) AS RN
        FROM CLUSTER_MEMBERS
        WHERE INSURANCE_ID IS NOT NULL
    )
    WHERE RN = 1
),
*/
/*
  ANY_TRUE fields — flags where one asserting source is sufficient
*/ 
/*
DECEASED_SURVIVED AS (
    SELECT
        ENTERPRISE_PATIENT_ID,
        BOOL_OR(DECEASED_FLAG)      AS IS_DECEASED,
        -- If deceased, take the earliest reported deceased date
        -- (the first source to report it is most likely to be accurate)
        MIN(CASE WHEN DECEASED_FLAG = TRUE THEN DECEASED_DATE END)
                                    AS DECEASED_DATE
    FROM CLUSTER_MEMBERS
    GROUP BY ENTERPRISE_PATIENT_ID
),
*/
/*
 CONFLICT COLLECTION
 Gather all flagged fields into a single ARRAY per cluster.
 This powers the GOLDEN_CONFLICTS output at the end.
*/
CONFLICT_FLAGS AS (
    SELECT
        COALESCE(
            fn.ENTERPRISE_PATIENT_ID,
            ln.ENTERPRISE_PATIENT_ID,
            dn.ENTERPRISE_PATIENT_ID,
            gn.ENTERPRISE_PATIENT_ID
        )                                           AS ENTERPRISE_PATIENT_ID,
        ARRAY_CONSTRUCT_COMPACT(
            CASE WHEN COALESCE(fn.FNAME_CONFLICTED,  FALSE)
                 THEN 'FIRST_NAME'  ELSE NULL END,
            CASE WHEN COALESCE(ln.LNAME_CONFLICTED,  FALSE)
                 THEN 'LAST_NAME'   ELSE NULL END,
            CASE WHEN COALESCE(dn.DOB_CONFLICTED,    FALSE)
                 THEN 'DATE_OF_BIRTH' ELSE NULL END,
            CASE WHEN COALESCE(gn.GENDER_CONFLICTED, FALSE)
                 THEN 'GENDER'      ELSE NULL END
        )                                           AS CONFLICTED_FIELDS,
        ARRAY_SIZE(ARRAY_CONSTRUCT_COMPACT(
            CASE WHEN COALESCE(fn.FNAME_CONFLICTED,  FALSE) THEN 1 ELSE NULL END,
            CASE WHEN COALESCE(ln.LNAME_CONFLICTED,  FALSE) THEN 1 ELSE NULL END,
            CASE WHEN COALESCE(dn.DOB_CONFLICTED,    FALSE) THEN 1 ELSE NULL END,
            CASE WHEN COALESCE(gn.GENDER_CONFLICTED, FALSE) THEN 1 ELSE NULL END
        ))                                          AS CONFLICT_COUNT
    FROM FNAME_SURVIVED   fn
    FULL OUTER JOIN LNAME_SURVIVED   ln USING (ENTERPRISE_PATIENT_ID)
    FULL OUTER JOIN DOB_SURVIVED     dn USING (ENTERPRISE_PATIENT_ID)
    FULL OUTER JOIN GENDER_SURVIVED  gn USING (ENTERPRISE_PATIENT_ID)
)
/*
 FINAL ASSEMBLY
 Join all survived fields into one golden record per patient.
 Fields that conflicted are NULL with the conflict logged in the
 CONFLICTED_FIELDS array. Stewards resolve conflicts via the
 STEWARD_QUEUE — the golden record is still written so downstream
 systems get a usable record immediately.
*/ 
SELECT
    -- Patient identity
    cs.ENTERPRISE_PATIENT_ID,
 
    -- Identity fields (MOST_FREQUENT) 
    fn.FIRST_NAME,
    --mn.MIDDLE_NAME,
    ln.LAST_NAME,
    --sx.NAME_SUFFIX,
    dn.DATE_OF_BIRTH,
    gn.GENDER,
    --rn.RACE,
    --en.ETHNICITY,
 
    -- Contact fields (MOST_RECENT) 
    ad.PATIENT_ADDRESS1       AS ADDRESS_LINE1,
    --ad.ADDRESS_LINE2_STD    AS ADDRESS_LINE2,
    ad.PATIENT_CITY           AS CITY,
    ad.PATIENT_STATE          AS STATE,
    ad.PATIENT_ZIPCODE        AS ZIPCODE,
    --ad.COUNTRY_STD          AS COUNTRY,
    --ph.PHONE_STD            AS PHONE_PRIMARY,
    --em.EMAIL_STD            AS EMAIL,
    --ins.INSURANCE_ID,
 
    -- Clinical flags (ANY_TRUE) 
    --dc.IS_DECEASED,
    --dc.DECEASED_DATE,
 
    -- Cluster metadata 
    cs.RECORD_COUNT,
    --cs.SOURCE_COUNT,
    ROUND(cs.AVG_CONFIDENCE, 4)     AS CONFIDENCE_SCORE,
    --si.SOURCE_IDENTIFIERS,
 
    /*
     Conflict metadata 
      CONFLICTED_FIELDS: array of field names that could not be resolved.
      Empty array = clean golden record, no steward action needed.
      Non-empty = golden record has NULLs that must be resolved manually.
    */
    cf.CONFLICTED_FIELDS,
    cf.CONFLICT_COUNT,
    CASE
        WHEN cf.CONFLICT_COUNT = 0 THEN 'CLEAN'
        WHEN cf.CONFLICT_COUNT <= 2 THEN 'PARTIAL_CONFLICT'
        ELSE                             'HIGH_CONFLICT'
    END                             AS GOLDEN_RECORD_STATUS,
 
    CURRENT_TIMESTAMP()             AS SURVIVED_AT
 
FROM CLUSTER_STATS          cs
LEFT JOIN FNAME_SURVIVED    fn  ON fn.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
-- LEFT JOIN MNAME_SURVIVED    mn  ON mn.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
LEFT JOIN LNAME_SURVIVED    ln  ON ln.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
--LEFT JOIN SUFFIX_SURVIVED   sx  ON sx.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
LEFT JOIN DOB_SURVIVED      dn  ON dn.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
LEFT JOIN GENDER_SURVIVED   gn  ON gn.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
--LEFT JOIN RACE_SURVIVED     rn  ON rn.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
--LEFT JOIN ETHNICITY_SURVIVED en ON en.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
LEFT JOIN ADDRESS_SURVIVED  ad  ON ad.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
--LEFT JOIN PHONE_SURVIVED    ph  ON ph.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
--LEFT JOIN EMAIL_SURVIVED    em  ON em.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
--LEFT JOIN INSURANCE_SURVIVED ins ON ins.ENTERPRISE_PATIENT_ID = cs.ENTERPRISE_PATIENT_ID
--LEFT JOIN DECEASED_SURVIVED dc  ON dc.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
LEFT JOIN SOURCE_ID_BAG     si  ON si.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
LEFT JOIN CONFLICT_FLAGS    cf  ON cf.ENTERPRISE_PATIENT_ID  = cs.ENTERPRISE_PATIENT_ID
 
ORDER BY cs.ENTERPRISE_PATIENT_ID;


In [ ]:
%%sql -r dataframe_30


## FINAL STATUS REPORT
-- All numbers should be reviewed before survivorship executes.

In [ ]:
%%sql -r dataframe_21
SELECT ' --PAIR INTAKE --' AS SECTION, NULL AS METRIC, NULL AS VALUE
UNION ALL
SELECT '', 'Total pairs ingested',
    COUNT(*)::VARCHAR FROM T_PAIR_INTAKE
UNION ALL
SELECT '', 'AUTO_MATCH pairs',
    COUNT_IF(MATCH_DECISION IN ('AUTO_MATCH','TRUE_MATCH')) AS MATCH_COUNT FROM T_PAIR_INTAKE
UNION ALL
SELECT '', 'REVIEW pairs',
    COUNT_IF(MATCH_DECISION = 'REVIEW')::VARCHAR FROM T_PAIR_INTAKE
UNION ALL
SELECT '', 'Already-clustered pairs (skipped)',
    COUNT_IF(CLUSTER_SITUATION = 'ALREADY_CLUSTERED')::VARCHAR FROM T_PAIR_INTAKE

UNION ALL SELECT '--CLUSTER RESOLUTION--', NULL, NULL
UNION ALL
SELECT '', 'New components formed',
    COUNT_IF(EPI_TYPE = 'NEW')::VARCHAR FROM T_EPI_ASSIGNMENT
UNION ALL
SELECT '', 'Existing clusters extended',
    COUNT_IF(EPI_TYPE = 'EXISTING')::VARCHAR FROM T_EPI_ASSIGNMENT

UNION ALL SELECT '--MERGES--', NULL, NULL
UNION ALL
SELECT '', 'Auto-approved merges',
    COUNT_IF(MERGE_APPROVAL = 'AUTO_APPROVE')::VARCHAR FROM T_MERGE_CANDIDATES
UNION ALL
SELECT '', 'Merges requiring steward sign-off',
    COUNT_IF(MERGE_APPROVAL = 'STEWARD_REQUIRED')::VARCHAR FROM T_MERGE_CANDIDATES

/*
UNION ALL SELECT '--STEWARD QUEUE--', NULL, NULL
UNION ALL
SELECT '', 'Pairs queued for steward review',
    COUNT(*)::VARCHAR FROM STEWARD_QUEUE WHERE DECISION IS NULL
*/ 

UNION ALL SELECT '--GOLDEN READINESS--', NULL, NULL
UNION ALL
SELECT '', 'Clusters ready for survivorship',
    COUNT_IF(READY_FOR_SURVIVORSHIP = TRUE)::VARCHAR  FROM GOLDEN_READINESS
UNION ALL
SELECT '', 'Clusters blocked from survivorship',
    COUNT_IF(READY_FOR_SURVIVORSHIP = FALSE)::VARCHAR FROM GOLDEN_READINESS

UNION ALL SELECT '--PROCEED?--', NULL, NULL
UNION ALL
SELECT '',
    CASE WHEN COUNT_IF(REQUIRES_REVIEW = TRUE) > 0
         THEN '⚠  HOLD — resolve ' || COUNT_IF(REQUIRES_REVIEW = TRUE)::VARCHAR
              || ' open merge events before running survivorship'
         ELSE '✓  CLEAR — run #5 survivorship on READY_FOR_SURVIVORSHIP = TRUE clusters'
    END,
    NULL
FROM MERGE_EVENTS
WHERE PERFORMED_BY = 'PENDING_STEWARD';

## #6  DATA QUALITY RULES
***Input:***  Standardized records from #1
***Output:*** One row per issue detected, with severity and field context

***Pattern:*** 
LATERAL FLATTEN over an ARRAY_CONSTRUCT of CASE expressions.

Each CASE either returns an issue object or NULL.
FLATTEN unpacks the array, and the WHERE clause drops NULLs.
This produces one output row per issue per record — multiple issues on one record produce multiple rows, all linkable by STD_RECORD_ID.

     Severity levels:
       CRITICAL — record cannot be matched reliably; route to steward
       HIGH     — significant field gap; degrades match quality
       MEDIUM   — corroborating field missing; minor match quality impact
       LOW      — cosmetic or low-signal field issue

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE DQ_ISSUES_LOG AS
SELECT
    DIM_PATIENT_KEY,
    ---SOURCE_SYSTEM_ID,
    CURRENT_TIMESTAMP() AS RESOLVED_AT,
    issue.value:ISSUE_CODE::VARCHAR        AS ISSUE_CODE,
    issue.value:ISSUE_SEVERITY::VARCHAR    AS ISSUE_SEVERITY,
    issue.value:FIELD_NAME::VARCHAR        AS FIELD_NAME,
    issue.value:FIELD_VALUE::VARCHAR       AS FIELD_VALUE,
    issue.value:ISSUE_DESCRIPTION::VARCHAR AS ISSUE_DESCRIPTION

FROM STAN_NORM,

LATERAL FLATTEN(INPUT => ARRAY_CONSTRUCT(

    -- Missing last name → HIGH
    CASE WHEN ln_std IS NULL
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'MISSING_LAST_NAME',
            'ISSUE_SEVERITY',    'HIGH',
            'FIELD_NAME',        'LN_STD',
            'FIELD_VALUE',       NULL,
            'ISSUE_DESCRIPTION', 'Last name is missing — record will not enter primary blocking pass')
        ELSE NULL END,

    -- Missing or unparseable DOB → CRITICAL
    CASE WHEN PATIENT_DOB IS NULL
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'MISSING_DOB',
            'ISSUE_SEVERITY',    'CRITICAL',
            'FIELD_NAME',        'PATIENT_DOB',
            'FIELD_VALUE',       NULL,
            'ISSUE_DESCRIPTION', 'DOB is missing or could not be parsed — highest-weight match field is zero')
        ELSE NULL END,

    -- DOB in the future → CRITICAL (data entry error or test record)
    CASE WHEN PATIENT_DOB IS NOT NULL
          AND PATIENT_DOB > CURRENT_DATE()
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'FUTURE_DOB',
            'ISSUE_SEVERITY',    'CRITICAL',
            'FIELD_NAME',        'PATIENT_DOB',
            'FIELD_VALUE',       PATIENT_DOB::VARCHAR,
            'ISSUE_DESCRIPTION', 'DOB is in the future — likely data entry error or test record')
        ELSE NULL END,

    -- Implausible age (> 130 years) → HIGH
    -- Common cause: default date 1900-01-01 used when DOB unknown
    CASE WHEN PATIENT_DOB IS NOT NULL
          AND DATEDIFF('year', PATIENT_DOB, CURRENT_DATE()) > 130
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'IMPLAUSIBLE_AGE',
            'ISSUE_SEVERITY',    'HIGH',
            'FIELD_NAME',        'PATIENT_DOB',
            'FIELD_VALUE',       PATIENT_DOB::VARCHAR,
            'ISSUE_DESCRIPTION', 'Age > 130 years — likely placeholder date (e.g. 1900-01-01)')
        ELSE NULL END,

    -- Invalid or missing gender code → MEDIUM
    CASE WHEN PATIENT_GENDER IS NULL OR PATIENT_GENDER NOT IN ('Male','Female','Unknown','Other')
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'INVALID_GENDER',
            'ISSUE_SEVERITY',    'MEDIUM',
            'FIELD_NAME',        'PATIENT_GENDER',
            'FIELD_VALUE',       PATIENT_GENDER,
            'ISSUE_DESCRIPTION', 'Gender code is missing or outside expected values Male/Female/Unknown/Other')
        ELSE NULL END,

    -- ZIP not exactly 5 digits → MEDIUM
    CASE WHEN PATIENT_ZIPCODE IS NOT NULL
          AND NOT REGEXP_LIKE(PATIENT_ZIPCODE, '^[0-9]{5}$')
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'INVALID_ZIP',
            'ISSUE_SEVERITY',    'MEDIUM',
            'FIELD_NAME',        'PATIENT_ZIPCODE',
            'FIELD_VALUE',       PATIENT_ZIPCODE,
            'ISSUE_DESCRIPTION', 'ZIP code is not a valid 5-digit value after standardization')
        ELSE NULL END,
    -- MISSING STATE
    CASE WHEN PATIENT_STATE IS NULL
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'MISSING_STATE',
            'ISSUE_SEVERITY',    'MEDIUM',
            'FIELD_NAME',        'PATIENT_ZIPCODE',
            'FIELD_VALUE',        PATIENT_STATE,
            'ISSUE_DESCRIPTION', 'Patient State Missing')
        ELSE NULL END,     
    -- MISSING CITY
    CASE WHEN PATIENT_CITY IS NULL
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'MISSING_CITY',
            'ISSUE_SEVERITY',    'MEDIUM',
            'FIELD_NAME',        'PATIENT_CITY',
            'FIELD_VALUE',        PATIENT_CITY,
            'ISSUE_DESCRIPTION', 'Patient CITY Missing')
        ELSE NULL END,
    -- MISSING ADDRESS1
    CASE WHEN PATIENT_ADDRESS1 IS NULL
        THEN OBJECT_CONSTRUCT(
            'ISSUE_CODE',        'MISSING_ADDRESS',
            'ISSUE_SEVERITY',    'MEDIUM',
            'FIELD_NAME',        'PATIENT_ADDRESS1',
            'FIELD_VALUE',        PATIENT_ADDRESS1,
            'ISSUE_DESCRIPTION', 'Patient ADDRESS Missing')
        ELSE NULL END
)) AS issue

WHERE issue.value IS NOT NULL;   -- drop the NULL entries (fields that passed their check)

In [ ]:
%%sql -r dataframe_34
SELECT ISSUE_DESCRIPTION,COUNT(*) AS FREQUENCY FROM DQ_QUALITY_ISSUES GROUP BY ISSUE_DESCRIPTION;

## #9  SINGLETON DETECTION AND ROUTING

Records that land in no block under any blocking key in #2 cannot be matched. They are not errors — they are a routing decision.

Route by reason:
NULL_KEY_FIELD   → DQ queue (fix the data, re-ingest)
RARE_DEMOGRAPHICS, high DQ → auto-provision as new master patient
RARE_DEMOGRAPHICS, low DQ  → steward review

Singletons are re-evaluated on every pipeline run — a singleton today may find a match when a second record from the same source arrives tomorrow. Do not permanently discard them.

***IDENTIFICATION APPROACH*** — aggregation, not joining:

A record is a singleton if it matches no other record on ANY blocking key. Instead of joining to find matches, aggregate each
blocking key independently to count how many records share each key value. A record is a singleton if its key value count = 1
 (unique value) OR its key is NULL (no value at all).

Each CTE below is a single GROUP BY scan — O(n) not O(n²).
The final join is each record against three small lookup tables, not against the full record table.

***Complexity:*** O(n) for each key aggregation → O(3n) total.

***Previous:***   O(n²) for the self-join.

On 1M records: ~3M row operations vs ~1 trillion. Order of magnitude improvement in practice even accounting for Snowflake parallelism.


In [ ]:
%%sql -r dataframe_4
WITH
 -- Step 1: Count records per blocking key value 
--   One GROUP BY per key. These are cheap full-table aggregations.
--   Result: for each distinct key value, how many records share it?
--   Key value with count = 1 means that value is unique → any record
--   holding it has no block-mates under this key.

PRIMARY_BLOCK AS (
-- LNSDX_ZIP
    SELECT
        ln_sdx || '|' || PATIENT_ZIPCODE::VARCHAR  AS KEY_VALUE,
        COUNT(*)                               AS KEY_COUNT
    FROM STAN_NORM
    WHERE 
          ln_sdx IS NOT NULL
      AND PATIENT_ZIPCODE IS NOT NULL
    GROUP BY 
        ln_sdx,
        PATIENT_ZIPCODE
),
 
FALLBACK_BLOCK AS (
-- LNSDX_DOB
    SELECT
        ln_sdx || '|' || PATIENT_DOB::VARCHAR   AS KEY_VALUE,
        COUNT(*)                                AS KEY_COUNT
    FROM STAN_NORM
    WHERE ln_sdx           IS NOT NULL
      AND PATIENT_DOB      IS NOT NULL
    GROUP BY ln_sdx, PATIENT_DOB
),
 
LAST_RESORT_BLOCK AS (
-- ZIP3_DOB
    SELECT
        PATIENT_DOB::VARCHAR || '|' || ZIP3        AS KEY_VALUE,
        COUNT(*)                                   AS KEY_COUNT
    FROM STAN_NORM
    WHERE PATIENT_DOB IS NOT NULL
      AND ZIP3        IS NOT NULL
    GROUP BY PATIENT_DOB, ZIP3
),
 
-- Step 2: For each record, look up whether any key has count > 1 
--   LEFT JOIN each record against the three count tables.
--   If the count for a key is NULL (no match = key was NULL on this record)
--   or = 1 (unique value = no other record shares this key), that key
--   cannot place the record in a block with anyone else.
--   A record is a singleton only if ALL THREE keys are NULL or unique.
 
KEYED AS (
    SELECT
        s.DIM_PATIENT_KEY,
        --s.SOURCE_SYSTEM_ID,
        s.ln_std,
        s.fn_std,
        s.PATIENT_DOB,
        s.PATIENT_ZIPCODE,
        --s.DQ_SCORE_OVERALL,
        s.ln_sdx,
        s.fn_sdx,
 
        -- LNSDX_ZIP
        pri.KEY_COUNT    AS LNSDX_ZIP_PEERS,
 
        -- LNSDX_DOB
        fb.KEY_COUNT    AS LNSDX_DOB_PEERS,
 
        -- ZIP3_DOB
        lr.KEY_COUNT    AS ZIP3_DOB_PEERS
 
    FROM STAN_NORM AS s
 
    LEFT JOIN PRIMARY_BLOCK AS pri
        ON pri.KEY_VALUE = s.ln_sdx || '|' || s.PATIENT_ZIPCODE::VARCHAR 
        AND s.ln_sdx           IS NOT NULL
        AND s.PATIENT_ZIPCODE  IS NOT NULL

    LEFT JOIN FALLBACK_BLOCK AS fb
        ON fb.KEY_VALUE = s.ln_sdx || '|' || s.PATIENT_DOB::VARCHAR
        AND s.ln_sdx IS NOT NULL
        AND s.PATIENT_DOB     IS NOT NULL
 
    LEFT JOIN LAST_RESORT_BLOCK AS lr
        ON lr.KEY_VALUE = s.PATIENT_DOB::VARCHAR || '|' || s.ZIP3
        AND s.PATIENT_DOB IS NOT NULL
        AND s.ZIP3        IS NOT NULL
)
 
SELECT
    DIM_PATIENT_KEY,
    --SOURCE_SYSTEM_ID,
    ln_std,
    fn_std,
    PATIENT_DOB,
    PATIENT_ZIPCODE,
    --DQ_SCORE_OVERALL,
 
    -- Expose peer counts for transparency 
    COALESCE(LNSDX_ZIP_PEERS, 0)        AS LNSDX_ZIP_PEERS,
    COALESCE(LNSDX_DOB_PEERS, 0)        AS LNSDX_DOB_PEERS,
    COALESCE(ZIP3_DOB_PEERS,  0)        AS ZIP3_DOB_PEERS,
 
    -- Classify the reason for being a singleton.
    -- Ordered from most actionable (fixable data problem) to least
    -- (genuinely rare demographics — not necessarily a data problem).
    CASE
        WHEN ln_std  IS NULL
         AND fn_std IS NULL                            THEN 'NULL_BOTH_NAMES'
        WHEN ln_std  IS NULL                           THEN 'NULL_LAST_NAME'
        WHEN fn_std IS NULL                            THEN 'NULL_FIRST_NAME'
        WHEN PATIENT_DOB  IS NULL                      THEN 'NULL_DOB'
        WHEN PATIENT_ZIPCODE IS NULL                   THEN 'NULL_ZIP'
        ELSE                                                'RARE_DEMOGRAPHICS'
    END AS SINGLETON_REASON,
 
    -- Routing decision
    CASE
        WHEN ln_std           IS NULL
          OR PATIENT_ZIPCODE  IS NULL
          OR PATIENT_DOB      IS NULL                            THEN 'DQ_QUEUE'
        --WHEN DQ_SCORE_OVERALL >= 70                            THEN 'AUTO_PROVISION_MPI'
        ELSE                                                          'STEWARD_REVIEW'
    END AS SUGGESTED_ROUTE,
 
    CURRENT_TIMESTAMP() AS DETECTED_AT
 
FROM KEYED
WHERE
    -- A record is a singleton if it has no block-mates under ANY key.
    -- No block-mates = key is NULL (LEFT JOIN returned nothing)
    --               OR key count = 1 (only this record holds that value)
    COALESCE(LNSDX_ZIP_PEERS, 0) <= 1
    AND COALESCE(LNSDX_DOB_PEERS,  0) <= 1
    AND COALESCE(ZIP3_DOB_PEERS,    0) <= 1;
 

# #10 GROUND TRUTH BOOTSTRAP
Need to assemble known match pairs without SSN, to measure PC in #7 and calibrate thresholds in #4.

Use the AUTO_MATCHED as initial potential 'bootstrap' pairs - then in place of intensive human assembly of cases by hand, leverage AI

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE BOOTSTRAP_TRUTH AS
WITH 
TRUTH_CHECK AS (
    SELECT 
        REC_A,REC_B,FINAL_SCORE,MATCH_DECISION,
    FROM  
        COMP_DECISION
    WHERE 
          MATCH_DECISION IN ('AUTO_MATCH','REVIEW') AND FINAL_SCORE<=0.90
)
SELECT TOP 5000
    REC_A,REC_B,FINAL_SCORE,MATCH_DECISION,
        a.LN_STD AS LN_STD_A,b.LN_STD AS LN_STD_B,
        a.FN_STD AS FN_STD_A,b.FN_STD AS FN_STD_B,
        a.PATIENT_DOB AS PATIENT_DOB_A,b.PATIENT_DOB AS PATIENT_DOB_B,
        a.PATIENT_GENDER AS PATIENT_GENDER_A,b.PATIENT_GENDER AS PATIENT_GENDER_b,
        a.PATIENT_CITY AS PATIENT_CITY_A, b.PATIENT_CITY AS PATIENT_CITY_B, 
        a.PATIENT_STATE AS PATIENT_STATE_A, b.PATIENT_STATE AS PATIENT_STATE_B, 
        a.PATIENT_ZIPCODE AS PATIENT_ZIPCODE_A, b.PATIENT_ZIPCODE PATIENT_ZIPCODE_B,
        a.MRN AS MRN_A, b.MRN AS MRN_b,
        a.IDENTIFIER AS IDENTIFIER_A, B.IDENTIFIER AS IDENTIFIER_B,
        a.PATIENT_ADDRESS1 AS PATIENT_ADDRESS1_A,b.PATIENT_ADDRESS1 AS PATIENT_ADDRESS1_B,
        a.MOBILE_NUMBER AS MOBILE_NUMBER_A,b.MOBILE_NUMBER AS MOBILE_NUMBER_B,
        AI_COMPLETE(
        '***llama3.1-70b****',  
        CONCAT(
            'You are helping deduplicate person records.\n\n',
            'Decide whether these two records refer to:\n',
            '(A) the same person\n',
            '(B) different people (including twins or siblings)\n',
            $$Default to '(B) different' when uncertain.\n$$,
            'Common names, shared households, twins, and siblings are real.\n',
            'Do not assume a match without clear discriminating evidence.\n\n',

            'KNOWN DATA QUALITY PATTERNS (apply when interpreting discrepancies):\n',
                '- First name: nicknames: A nickname explanation is only valid when the two names share a well-established diminutive relationship for the SAME root name;
                   transliteration and typos: are common and generally do not support difference alone, but are differnent from nicknames.\n',
                '- Last name: maiden/married name changes, hyphenation, cultural ordering differences, ',
                    'ignore suffixes in determining difference when DOB matches.\n',
                '- Address: changes are expected across multi-year record spans, and are more likely relocations when names and DOB match.\n',
                '- Gender: recording inconsistency exists across source systems (biological sex vs. gender identity). Treat as weak signal.\n',
                '- Mobile Number: Ignore number when missing for Record 1 or Record 2, otherwise the same number overrides street address differences for sameness.\n\n',

            'Record 1:\n',
            '- Name: ', FN_STD_A, ' ', LN_STD_A, '\n',
            '- DOB: ', COALESCE(TO_VARCHAR(PATIENT_DOB_A), 'Unknown'), '\n',
            '- Gender: ', COALESCE(PATIENT_GENDER_A, 'Unknown'), '\n',
            '- Address: ', COALESCE(PATIENT_ADDRESS1_A, 'Unknown'), '\n',
            '- City: ', COALESCE(PATIENT_CITY_A, 'Unknown'), '\n',
            '- State: ', COALESCE(PATIENT_STATE_A, 'Unknown'), '\n',
            '- Zipcode: ', COALESCE(PATIENT_ZIPCODE_A, 'Unknown'), '\n',
            '- MobilePhone: ',COALESCE(MOBILE_NUMBER_A, 'Unknown'), '\n',

            'Record 2:\n',
            '- Name: ', FN_STD_B, ' ', LN_STD_B, '\n',
            '- DOB: ', COALESCE(TO_VARCHAR(PATIENT_DOB_B), 'Unknown'), '\n',
            '- Gender: ', COALESCE(PATIENT_GENDER_B, 'Unknown'), '\n',
            '- Address: ', COALESCE(PATIENT_ADDRESS1_B, 'Unknown'), '\n',
            '- City: ', COALESCE(PATIENT_CITY_B, 'Unknown'), '\n\n',
            '- State: ', COALESCE(PATIENT_STATE_B, 'Unknown'), '\n',
            '- Zipcode: ', COALESCE(PATIENT_ZIPCODE_B, 'Unknown'), '\n',
            '- MobilePhone: ',COALESCE(MOBILE_NUMBER_B, 'Unknown'), '\n\n',

            'Think carefully about whether people can share these attributes.\n',

            $$
             Answer with:
             Decision: A(Same) or B(Different); Confidence: 0–1; Reasoning: short explanation;'
            $$
        )
    ) AS AI_DECISION

FROM TRUTH_CHECK AS chk
    JOIN STAN_NORM AS a ON a.DIM_PATIENT_KEY = chk.REC_A
    JOIN STAN_NORM AS b ON b.DIM_PATIENT_KEY = chk.REC_B 
HAVING CONTAINS(ai_decision,'A(Same)')

In [ ]:
%%sql -r BOOTSTRAP_TRUTH
SELECT * FROM BOOTSTRAP_TRUTH;

In [ ]:
%%sql -r dataframe_22
CREATE OR REPLACE TEMPORARY TABLE STEWARD_AI AS
WITH 
REVIEW_CHECK AS (
    SELECT 
        REC_A,REC_B,FINAL_SCORE
    FROM  
        COMP_DECISION
    WHERE 
          MATCH_DECISION IN ('REVIEW')
)
SELECT 
    REC_A,REC_B,FINAL_SCORE,
        a.LN_STD AS LN_STD_A,b.LN_STD AS LN_STD_B,
        a.FN_STD AS FN_STD_A,b.FN_STD AS FN_STD_B,
        a.PATIENT_DOB AS PATIENT_DOB_A,b.PATIENT_DOB AS PATIENT_DOB_B,
        a.PATIENT_GENDER AS PATIENT_GENDER_A,b.PATIENT_GENDER AS PATIENT_GENDER_B,
        a.PATIENT_CITY AS PATIENT_CITY_A, b.PATIENT_CITY AS PATIENT_CITY_B, 
        a.PATIENT_STATE AS PATIENT_STATE_A, b.PATIENT_STATE AS PATIENT_STATE_B, 
        a.PATIENT_ZIPCODE AS PATIENT_ZIPCODE_A, b.PATIENT_ZIPCODE AS PATIENT_ZIPCODE_B,
        a.MRN AS MRN_A, b.MRN AS MRN_b,
        a.IDENTIFIER AS IDENTIFIER_A, B.IDENTIFIER AS IDENTIFIER_B,
        a.PATIENT_ADDRESS1 AS PATIENT_ADDRESS1_A,b.PATIENT_ADDRESS1 AS PATIENT_ADDRESS1_B,
        a.MOBILE_NUMBER AS MOBILE_NUMBER_A,b.MOBILE_NUMBER AS MOBILE_NUMBER_B,
        AI_COMPLETE(
        '****llama3.1-70b****',  
        CONCAT(
            'You are helping deduplicate person records.\n\n',
            'Decide whether these two records refer to:\n',
            '(A) the same person\n',
            '(B) different people (including twins or siblings)\n',
            $$Default to '(B) different' when uncertain.\n$$,
            'Common names, shared households, twins, and siblings are real.\n',
            'Do not assume a match without clear discriminating evidence.\n\n',

            'KNOWN DATA QUALITY PATTERNS (apply when interpreting discrepancies):\n',
                '- First name: nicknames: A nickname explanation is only valid when the two names share a well-established diminutive relationship for the SAME root name;
                   transliteration and typos: are common and generally do not support difference alone, but are differnent from nicknames.\n',
                '- Last name: maiden/married name changes, hyphenation, cultural ordering differences, ',
                    'ignore suffixes in determining difference when DOB matches.\n',
                '- Address: changes are expected across multi-year record spans, and are more likely relocations when names and DOB match.\n',
                '- Gender: recording inconsistency exists across source systems (biological sex vs. gender identity). Treat as weak signal.\n',
                '- Mobile Number: Ignore number when missing for Record 1 or Record 2, otherwise the same number overrides street address differences for sameness.\n\n',

            'Record 1:\n',
            '- Name: ', FN_STD_A, ' ', LN_STD_A, '\n',
            '- DOB: ', COALESCE(TO_VARCHAR(PATIENT_DOB_A), 'Unknown'), '\n',
            '- Gender: ', COALESCE(PATIENT_GENDER_A, 'Unknown'), '\n',
            '- Address: ', COALESCE(PATIENT_ADDRESS1_A, 'Unknown'), '\n',
            '- City: ', COALESCE(PATIENT_CITY_A, 'Unknown'), '\n',
            '- State: ', COALESCE(PATIENT_STATE_A, 'Unknown'), '\n',
            '- Zipcode: ', COALESCE(PATIENT_ZIPCODE_A, 'Unknown'), '\n',
            '- MobilePhone: ',COALESCE(MOBILE_NUMBER_A, 'Unknown'), '\n',

            'Record 2:\n',
            '- Name: ', FN_STD_B, ' ', LN_STD_B, '\n',
            '- DOB: ', COALESCE(TO_VARCHAR(PATIENT_DOB_B), 'Unknown'), '\n',
            '- Gender: ', COALESCE(PATIENT_GENDER_B, 'Unknown'), '\n',
            '- Address: ', COALESCE(PATIENT_ADDRESS1_B, 'Unknown'), '\n',
            '- City: ', COALESCE(PATIENT_CITY_B, 'Unknown'), '\n\n',
            '- State: ', COALESCE(PATIENT_STATE_B, 'Unknown'), '\n',
            '- Zipcode: ', COALESCE(PATIENT_ZIPCODE_B, 'Unknown'), '\n',
            '- MobilePhone: ',COALESCE(MOBILE_NUMBER_B, 'Unknown'), '\n\n',

            'Think carefully about whether people can share these attributes.\n',

            $$
             Answer with:
             Decision: A(Same) or B(Different); Confidence: 0–1; Reasoning: short explanation;'
            $$
        )
    ) AS AI_DECISION
    
FROM REVIEW_CHECK AS chk
    JOIN STAN_NORM AS a ON a.DIM_PATIENT_KEY = chk.REC_A
    JOIN STAN_NORM AS b ON b.DIM_PATIENT_KEY = chk.REC_B 

In [ ]:
%%sql -r dataframe_32
SELECT * FROM STEWARD_AI WHERE CONTAINS(AI_DECISION,'(Same)');
--SELECT FINAL_SCORE,COUNT(*) AS FREQ FROM STEWARD_AI GROUP BY FINAL_SCORE;

In [ ]:
%%sql -r dataframe_39
WITH PARSE_DECISION AS(
    SELECT 
        CASE
          WHEN CONTAINS(AI_DECISION,'(Different)')
          THEN 'NO Match'
          WHEN CONTAINS(AI_DECISION,'(Same)')
          THEN 'MATCH'
          ELSE '!!!BAD!!!'
          END AS DECISION
    FROM STEWARD_AI 
)
SELECT 
    DECISION,
    COUNT(*) AS FREQUENCY
FROM
    PARSE_DECISION
GROUP BY
    DECISION;

In [ ]:
%%sql -r dataframe_40
WITH PARSE_DECISION AS(
    SELECT 
        *,
        CASE
          WHEN CONTAINS(AI_DECISION,'(Different)')
          THEN 'NO Match'
          WHEN CONTAINS(AI_DECISION,'(Same)')
          THEN 'MATCH'
          ELSE '!!!BAD!!!'
          END AS DECISION
    FROM STEWARD_AI 
)
SELECT 
    *
FROM
    PARSE_DECISION
WHERE
    DECISION='!!!BAD!!!';